# BTFuseViT v6 — leakage-controlled evaluation

This notebook trains and evaluates BTFuseViT under a **leakage-controlled evaluation protocol** and produces every
number, table and figure in the paper. The model and training recipe are the same as v5
(ViT-B/16 @ 384, MLP head, 3-phase training with ASL + SWA, temperature scaling, K = 14 TTA).
What changes is the **data protocol and the measurements**:

| Topic | What this notebook does |
|---|---|
| Validation / test separation | A separate **validation split** (grouped, from the training pool) is used for checkpoint selection, the SWA decision and temperature fitting. The **test sets are used once**, at the end. |
| Patient-level leakage | LGG is used for training only (grouped by TCGA patient). Figshare-derived Nickparvar/BRISC images are traced to **patient IDs** (Cheng et al. `.mat` files); validation is grouped by patient/duplicate cluster; seen-vs-unseen-patient accuracy is reported and a **patient-disjoint** run is trained. |
| Overlap between datasets | **Duplicate screen**: every image pair across Nickparvar-train/test, BRISC-train/test, LGG and PMRAM is compared (raw + band-pass normalised cross-correlation of brain-cropped thumbnails, flip-aware). Training images that are copies of, or closely related to (e.g. an adjacent slice), any test image are **removed**; exact copies within the training pool are de-duplicated. |
| BRISC / external evaluation | Full test sets (Nickparvar v2 *Testing*, all 1,600; BRISC *test*, all 1,000) + **PMRAM** (Bangladesh, 1,600 raw images). Part of PMRAM also occurs in the public datasets, so results are reported for all PMRAM images and for the PMRAM images with **no match** in any public source. Per-class metrics with 95% CIs for every test set. |
| Comparison with published work | No statistical test against published numbers; McNemar is used only between our own configurations on identical test images. |
| Controlled ablations | Every ablation uses the **identical** schedule, selection rule, calibration and TTA: −LGG, −BRISC, Nickparvar-only, no glioma boost, no weighted sampling, percentile vs White Stripe normalisation; SWA / temperature / TTA are ablated from the same trained model. |
| TTA vs temperature, calibration | Single-view, single-view+T, TTA, TTA+T accuracy; **ECE, Brier, NLL**; reliability diagrams; per-view accuracy. |
| Grad-CAM++ localisation | **Quantitative localisation** against BRISC expert tumour masks (pointing game, energy-in-mask, enrichment, IoU), a centre-bias baseline, and **misclassified cases**. |
| Compute | Params, **GFLOPs per pass and per image with K = 14**, peak GPU memory, latency (1 view vs 14 views), plus the same measurements for the comparator backbones. |
| T2-FLAIR vs T1-CE shortcut | LGG enters as **single-channel FLAIR** (no colour cue), tumour-free LGG slices are **not** labelled glioma; a **shortcut probe** measures how often tumour-free FLAIR slices are called "glioma" with vs without LGG in training. |

**How to run** — one pass, start to end
1. Runtime → Change runtime type → GPU (RTX PRO 6000 ≈ 3–4 h; an A100 also works).
2. **Run all.** Nothing needs to be edited. (Set `SMOKE_TEST = True` in the config cell for a ~10-minute code check on a small subset.)
3. At the end the notebook writes one file, `MyDrive/BTFuseViT_v6/BTFuseViT_v6_outputs.zip`, with every table (LaTeX), figure and number of the paper.

If Colab disconnects, reconnect and **Run all** again: every finished training is saved on Drive and is skipped.

In [ ]:
# ── 1. Dependencies (torch / torchvision are preinstalled on Colab) ─────────────────────────────
import subprocess, sys
def pip(pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
pip(['kagglehub', 'timm', 'grad-cam', 'h5py', 'statsmodels', 'opencv-python-headless', 'seaborn'])
print('✓ Dependencies installed')

✓ Dependencies installed


In [ ]:
# ── 2. Imports (numpy stack only — torch is imported later, after the CPU-parallel preprocessing) ──
import os, sys, gc, json, math, time, random, shutil, hashlib, warnings, platform, zipfile, re
import multiprocessing as mp
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd
import cv2
from PIL import Image
from scipy import stats
from scipy.fft import dct
from scipy.signal import find_peaks
from scipy.stats import binomtest
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import (confusion_matrix, roc_auc_score, roc_curve,
                             precision_recall_fscore_support)
from sklearn.model_selection import StratifiedGroupKFold

warnings.filterwarnings('ignore')
cv2.setNumThreads(1)

# ════════════════════════════════════════════════════════════════════════════════════════════════
SMOKE_TEST = False    # full experiment (True = quick code check on a tiny subset only)
# ════════════════════════════════════════════════════════════════════════════════════════════════

CFG = dict(
    # model / training recipe — identical to v5
    backbone='vit_base_patch16_384', img_size=384, num_classes=4, drop_rate=0.3,
    lr_head=4e-5, lr_backbone=5e-6, lr_warmup=1e-6, max_norm=0.5, pct_start=0.15,
    warmup_epochs=5, finetune_epochs=25, swa_start=20, swa_lr=1e-6, swa_anneal_epochs=5,
    weight_decay=1e-4, batch_size=8, eval_batch_size=32,
    num_workers=min(8, max(2, (os.cpu_count() or 4) - 2)),   # v5 used 4; the loader, not the GPU, limited v5's speed
    glioma_boost=1.5, lgg_crops_per=3, lgg_jitter=25, lgg_margin=40,
    tta_k=14, tta_seed=1234,
    # protocol
    val_frac=0.12, split_seed=42,
    # White Stripe (Shinohara et al. 2014): mode of the foreground KDE, ±width quantile stripe
    ws_width=0.05, ws_tail_pct=96.0, ws_window_pct=(0.5, 99.5), ws_min_fg_frac=0.05, ws_max_kde=20000,
    # duplicate / overlap screening
    # every pair with raw NCC >= pair_raw_min is kept for analysis; 'related' (primary) = raw >= dup_ncc and
    # band-pass >= dup_bp (copies, re-crops, adjacent slices); 'exact copy' = band-pass >= dup_exact_bp
    pair_raw_min=0.80, dup_ncc=0.85, dup_bp=0.85, dup_exact_bp=0.92, dup_bp_grid=(0.80, 0.85, 0.92, 0.97), desc_size=64,
    # statistics
    bootstrap_b=2000, ece_bins=15, stat_seed=42,
    # Grad-CAM++
    cam_pg_tolerance_px=15, cam_batch=16,
    # compute benchmark
    bench_warmup=20, bench_iters=100, bench_cpu_iters=5, comparator_size=224,
    # optional data
    use_figshare_pid=True, use_external_pmram=True,
    n_proc=max(1, (os.cpu_count() or 2) - 1),
)
SMOKE_PER_CLASS = 24          # images per class per source in SMOKE_TEST
SMOKE_LGG_PATIENTS = 4
if SMOKE_TEST:
    CFG.update(warmup_epochs=1, finetune_epochs=2, swa_start=1, swa_anneal_epochs=1,
               bootstrap_b=200, bench_iters=10, bench_warmup=3, bench_cpu_iters=1)

CLASSES = ['Glioma', 'Meningioma', 'Pituitary', 'No Tumor']
K6_VIEWS = [0, 1, 2, 4, 9, 13]   # identity, h-flip, v-flip, rot90, brightness, blur (subset quoted in the paper)
TTA_NAMES = ['Identity', 'H-flip', 'V-flip', 'Resize 448 + centre crop', r'Rot 90$^\circ$', r'Rot 180$^\circ$',
             r'Rot 270$^\circ$', r'Rot $\pm$15$^\circ$ (random)', 'Resize 416 + random crop', r'Brightness $\pm$0.2 (random)',
             r'Contrast $\pm$0.2 (random)', 'Zoom-out 0.9 + pad', r'H-flip + rot 90$^\circ$', r'Gaussian blur $\sigma=1$']   # LaTeX-safe

# ── Paths: persistent results on Google Drive, fast image cache on local disk ─────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST, LOCAL = Path('/content/drive/MyDrive/BTFuseViT_v6'), Path('/content/btf_cache')
else:
    PERSIST, LOCAL = Path('./BTFuseViT_v6'), Path('./btf_cache')
if SMOKE_TEST:
    PERSIST, LOCAL = PERSIST.with_name(PERSIST.name + '_SMOKE'), LOCAL.with_name(LOCAL.name + '_SMOKE')
DIRS = {k: PERSIST / k for k in ['manifests', 'runs', 'results', 'figures', 'generated', 'checkpoints']}
for d in list(DIRS.values()) + [LOCAL]:
    d.mkdir(parents=True, exist_ok=True)

# ── Run plan (each training ≈ 30 min on an RTX PRO 6000; set enabled=False to skip) ───────────────
BASE_RUN = dict(sources=('nick', 'lgg', 'brisc'), sampler='balanced', boost=CFG['glioma_boost'],
                norm='ws', seed=42, patient_policy='report', enabled=True)
RUN_PLAN = [
    dict(BASE_RUN, name='main', label='Full model (BTFuseViT)'),
    dict(BASE_RUN, name='abl_no_lgg', sources=('nick', 'brisc'), label='Without LGG crops'),
    dict(BASE_RUN, name='abl_no_brisc', sources=('nick', 'lgg'), label='Without BRISC-train'),
    dict(BASE_RUN, name='abl_nick_only', sources=('nick',), label='Nickparvar-train only'),
    dict(BASE_RUN, name='abl_no_boost', boost=1.0, label='Without glioma boost (class-balanced only)'),
    dict(BASE_RUN, name='abl_uniform', sampler='shuffle', boost=1.0, label='Without weighted sampling (uniform)'),
    dict(BASE_RUN, name='abl_percentile', norm='percentile', label='Percentile instead of White Stripe'),
    dict(BASE_RUN, name='main_patient_disjoint', patient_policy='exclude',
         label='Patient-disjoint training (figshare PIDs)'),
    dict(BASE_RUN, name='main_seed1', seed=1, label='Full model, seed 1', enabled=False),
    dict(BASE_RUN, name='main_seed2', seed=2, label='Full model, seed 2', enabled=False),
]
if SMOKE_TEST:   # two short trainings are enough to exercise every code path
    RUN_PLAN = [dict(r) for r in RUN_PLAN if r['name'] in ('main', 'abl_no_lgg')]

def seed_everything(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    except ImportError:
        pass
seed_everything(42)

def save_json(obj, path):
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))

print(f"✓ Config ready | SMOKE_TEST={SMOKE_TEST} | results → {PERSIST} | cache → {LOCAL} | CPU workers={CFG['n_proc']}")

Mounted at /content/drive
✓ Config ready | SMOKE_TEST=False | results → /content/drive/MyDrive/BTFuseViT_v6 | cache → /content/btf_cache | CPU workers=47


In [ ]:
# ── 3. Download datasets (kagglehub; Colab cache is used when available) ───────────────────────────
import kagglehub

def kdl(handle, required=True):
    try:
        p = Path(kagglehub.dataset_download(handle))
        print(f'✓ {handle:55s} → {p}')
        return p
    except Exception as e:
        msg = f'could not download {handle}: {e}'
        if required:
            raise RuntimeError(msg)
        print('⚠', msg)
        return None

P_NICK  = kdl('masoudnickparvar/brain-tumor-mri-dataset')   # v2 (Feb 2026): 5,600 train / 1,600 test
P_LGG   = kdl('mateuszbuda/lgg-mri-segmentation')           # TCGA-LGG, 110 patients, pre/FLAIR/post-contrast tif
P_BRISC = kdl('briscdataset/brisc2025')                      # derived from Nickparvar v1 (Fateh et al., Sci Data 2026)
P_FIG   = kdl('ashkhagan/figshare-brain-tumor-dataset', required=False) if CFG['use_figshare_pid'] else None
P_PMRAM = kdl('orvile/pmram-bangladeshi-brain-cancer-mri-dataset', required=False) if CFG['use_external_pmram'] else None

100%|██████████| 157M/157M [00:04<00:00, 35.5MB/s]

Extracting files...


✓ masoudnickparvar/brain-tumor-mri-dataset                → /root/.cache/kagglehub/datasets/masoudnickparvar/brain-tumor-mri-dataset/versions/2


100%|██████████| 714M/714M [00:19<00:00, 38.3MB/s]

Extracting files...


...

Extracting files...


✓ ashkhagan/figshare-brain-tumor-dataset                  → /root/.cache/kagglehub/datasets/ashkhagan/figshare-brain-tumor-dataset/versions/1


100%|██████████| 161M/161M [00:04<00:00, 36.0MB/s]

Extracting files...


✓ orvile/pmram-bangladeshi-brain-cancer-mri-dataset       → /root/.cache/kagglehub/datasets/orvile/pmram-bangladeshi-brain-cancer-mri-dataset/versions/2


In [ ]:
# ── 4. Unified manifest of every image (one row per raw image) ──────────────────────────────────────
IMG_EXT = {'.jpg', '.jpeg', '.png', '.tif', '.tiff', '.bmp'}

def class_from_name(name):
    s = name.lower().replace('-', ' ').replace('_', ' ')
    if 'glioma' in s: return 0
    if 'mening' in s: return 1
    if 'pituit' in s: return 2
    if 'notumor' in s.replace(' ', '') or 'no tumor' in s or 'normal' in s or 'healthy' in s: return 3
    return None

def find_dir(root, name):
    """First directory under root whose name equals `name` (case-insensitive), shallowest first."""
    if root is None: return None
    root = Path(root)
    if root.name.lower() == name.lower(): return root
    hits = sorted((p for p in root.rglob('*') if p.is_dir() and p.name.lower() == name.lower()),
                  key=lambda p: len(p.parts))
    return hits[0] if hits else None

def class_dir_images(split_dir):
    out = []
    for cd in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        lab = class_from_name(cd.name)
        if lab is None: continue
        for p in sorted(cd.rglob('*')):
            if p.suffix.lower() in IMG_EXT: out.append((p, lab))
    return out

def plane_from_brisc(name):
    m = re.search(r'_(ax|co|sa)_', name)
    return {'ax': 'axial', 'co': 'coronal', 'sa': 'sagittal'}[m.group(1)] if m else ''

def build_manifest():
    rows = []
    def add(source, path, label, root, **kw):
        rows.append(dict(uid=f'{source}:{Path(path).relative_to(root).as_posix()}', source=source,
                         path=str(path), label=int(label), plane=kw.get('plane', ''), pid=kw.get('pid', ''),
                         mask_path=kw.get('mask_path', ''), lgg_tumor=kw.get('lgg_tumor', False),
                         contrast=kw.get('contrast', 'T1')))
    # Nickparvar v2
    for split, src in [('Training', 'nick_train'), ('Testing', 'nick_test')]:
        d = find_dir(P_NICK, split)
        assert d is not None, f'Nickparvar {split} folder not found under {P_NICK}'
        for p, lab in class_dir_images(d): add(src, p, lab, P_NICK)
    # BRISC 2025 (classification task; masks from segmentation task share the file stem)
    bcls = find_dir(P_BRISC, 'classification_task'); bseg = find_dir(P_BRISC, 'segmentation_task')
    assert bcls is not None, f'BRISC classification_task not found under {P_BRISC}'
    for split, src in [('train', 'brisc_train'), ('test', 'brisc_test')]:
        mask_dir = (bseg / split / 'masks') if bseg is not None else None
        for p, lab in class_dir_images(bcls / split):
            mp_ = mask_dir / (p.stem + '.png') if mask_dir is not None else None
            add(src, p, lab, P_BRISC, plane=plane_from_brisc(p.name),
                mask_path=str(mp_) if (mp_ is not None and mp_.exists()) else '')
    # LGG (TCGA): every slice with its mask; FLAIR channel is used downstream
    kroot = find_dir(P_LGG, 'kaggle_3m')
    assert kroot is not None, f'LGG kaggle_3m not found under {P_LGG}'
    for pdir in sorted(p for p in kroot.iterdir() if p.is_dir()):
        for ip in sorted(pdir.glob('*.tif')):
            if ip.stem.endswith('_mask'): continue
            mpth = ip.with_name(ip.stem + '_mask.tif')
            if not mpth.exists(): continue
            tumor = bool(np.array(Image.open(mpth)).max() > 0)
            add('lgg', ip, 0, P_LGG, pid=pdir.name, mask_path=str(mpth), lgg_tumor=tumor,
                contrast='T2', plane='axial')
    # PMRAM external set — raw (non-augmented) images only
    if P_PMRAM is not None:
        cand = [d for d in P_PMRAM.rglob('*') if d.is_dir() and class_from_name(d.name) is not None
                and 'raw' in d.as_posix().lower() and 'augment' not in d.as_posix().lower()]
        if len({class_from_name(d.name) for d in cand}) == 4:
            for d in sorted(cand):
                for p in sorted(d.rglob('*')):
                    if p.suffix.lower() in IMG_EXT: add('pmram', p, class_from_name(d.name), P_PMRAM)
        else:
            print('⚠ PMRAM raw folders not found — external test set skipped (augmented copies are NOT used for testing)')
    df = pd.DataFrame(rows)
    if SMOKE_TEST:
        keep_pat = sorted(df.loc[df.source == 'lgg', 'pid'].unique())[:SMOKE_LGG_PATIENTS]
        parts = [df[(df.source == 'lgg') & df.pid.isin(keep_pat)]]
        for (s, l), g in df[df.source != 'lgg'].groupby(['source', 'label']):
            parts.append(g.sort_values('uid').head(SMOKE_PER_CLASS))
        df = pd.concat(parts, ignore_index=True)
    return df.reset_index(drop=True)

MAN = build_manifest()
MAN.to_csv(DIRS['manifests'] / 'manifest_raw.csv', index=False)
cnt = MAN.groupby(['source', 'label']).size().unstack(fill_value=0).rename(columns=dict(enumerate(CLASSES)))
print(cnt); print('LGG patients:', MAN.loc[MAN.source == 'lgg', 'pid'].nunique(),
                  '| tumour-bearing LGG slices:', int(MAN.lgg_tumor.sum()),
                  '| BRISC-test images with masks:', int((MAN.source.eq('brisc_test') & MAN.mask_path.ne('')).sum()))
if not SMOKE_TEST:
    n_tr, n_te = (MAN.source == 'nick_train').sum(), (MAN.source == 'nick_test').sum()
    if (n_tr, n_te) != (5600, 1600):
        print(f'⚠⚠ Nickparvar counts are {n_tr}/{n_te}, expected 5,600/1,600 (v2). '
              'The paper assumes v2 — check the results if the Kaggle version changed.')

label        Glioma  Meningioma  Pituitary  No Tumor
source                                              
brisc_test      254         306        300       140
brisc_train    1147        1329       1457      1067
lgg            3929           0          0         0
nick_test       400         400        400       400
nick_train     1400        1400       1400      1400
pmram           373         363        373       396
LGG patients: 110 | tumour-bearing LGG slices: 1373 | BRISC-test images with masks: 860


In [ ]:
# ── 5. Image helpers: reading, foreground mask, White Stripe, duplicate descriptors ───────────────
def read_gray(path, flair=False):
    """Float32 grayscale image. For LGG .tif files (channels = pre-contrast, FLAIR, post-contrast) the FLAIR
    channel is returned, so LGG enters the model as single-channel FLAIR (no colour cue)."""
    if flair:
        a = np.array(Image.open(path))
        if a.ndim == 3: a = a[..., 1]
        return a.astype(np.float32)
    return np.array(Image.open(path).convert('L')).astype(np.float32)

def to_u8(a):
    a = a.astype(np.float32); lo, hi = float(a.min()), float(a.max())
    return np.zeros(a.shape, np.uint8) if hi - lo < 1e-6 else ((a - lo) / (hi - lo) * 255).astype(np.uint8)

def foreground_mask(a):
    """Head/brain foreground: Otsu on a blurred copy → largest component → closing → hole filling."""
    u8 = cv2.GaussianBlur(to_u8(a), (5, 5), 0)
    thr, _ = cv2.threshold(u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = (u8 > max(thr, 1)).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n > 1:
        m = (lab == 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((9, 9), np.uint8))
    if m[0, 0] == 0:                                   # fill interior holes (e.g. dark ventricles/CSF)
        ff = m.copy(); cv2.floodFill(ff, np.zeros((m.shape[0] + 2, m.shape[1] + 2), np.uint8), (0, 0), 1)
        m = m | (ff == 0).astype(np.uint8)
    return m.astype(bool)

def whitestripe_stats(a, contrast='T1', width=None, tail_pct=None, seed=0):
    """White Stripe (Shinohara et al., NeuroImage Clin 2014) on a 2-D slice.
    T1: last (highest-intensity) mode of the foreground KDE after removing the upper tail;
    T2/FLAIR: largest mode. The stripe is the set of foreground pixels whose intensity quantile lies within
    ±width of the mode's quantile; returns its mean/SD (None → caller falls back to percentile scaling)."""
    width = CFG['ws_width'] if width is None else width
    tail_pct = CFG['ws_tail_pct'] if tail_pct is None else tail_pct
    m = foreground_mask(a)
    if m.mean() < CFG['ws_min_fg_frac']: return None
    v = a[m].astype(np.float64)
    rng = np.random.default_rng(seed)
    vs = rng.choice(v, CFG['ws_max_kde'], replace=False) if v.size > CFG['ws_max_kde'] else v
    vk = vs[vs <= np.percentile(vs, tail_pct)] if contrast == 'T1' else vs
    if vk.size < 100 or np.ptp(vk) < 1e-6: return None
    grid = np.linspace(vk.min(), vk.max(), 512)
    dens = stats.gaussian_kde(vk, bw_method='silverman')(grid)
    peaks, _ = find_peaks(dens, prominence=0.05 * dens.max())
    if len(peaks) == 0: peaks = np.array([int(np.argmax(dens))])
    mode = grid[peaks[-1]] if contrast == 'T1' else grid[peaks[int(np.argmax(dens[peaks]))]]
    q = float(np.mean(v < mode))
    lo, hi = np.quantile(v, [max(q - width, 0.0), min(q + width, 1.0)])
    ws = v[(v >= lo) & (v <= hi)]
    if ws.size < 50 or ws.std() < 1e-6: return None
    return dict(mu=float(ws.mean()), sd=float(ws.std()), mode=float(mode), fg=float(m.mean()))

def percentile_u8(a):
    """v5 behaviour: per-image p1–p99 clipping + min–max (also the White Stripe fallback)."""
    p1, p99 = np.percentile(a, [1, 99])
    return to_u8(np.clip(a, p1, p99))

def ws_to_u8(a, st, window):
    """White Stripe z-scores mapped with a FIXED, dataset-wide window (not per-image min–max, which would undo
    the standardisation: min–max of (x−μ)/σ equals min–max of x)."""
    z = (a - st['mu']) / st['sd']
    lo, hi = window
    return (np.clip((z - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)

# duplicate descriptors ----------------------------------------------------------------------------
_POP8 = np.array([bin(i).count('1') for i in range(256)], np.uint8)
def popcount64(x):
    if hasattr(np, 'bitwise_count'): return np.bitwise_count(x).astype(np.uint8)
    return _POP8[x.view(np.uint8)].reshape(x.shape + (8,)).sum(-1).astype(np.uint8)

def brain_crop(a):
    m = foreground_mask(a)
    ys, xs = np.nonzero(m)
    if ys.size < 0.02 * a.size: return a
    return a[ys.min():ys.max() + 1, xs.min():xs.max() + 1]

def phash64(g):
    """64-bit perceptual hash (DCT of a 32×32 thumbnail, 8×8 low frequencies vs their median)."""
    s = cv2.resize(g.astype(np.float32), (32, 32), interpolation=cv2.INTER_AREA)
    d = dct(dct(s, axis=0, norm='ortho'), axis=1, norm='ortho')[:8, :8]
    return np.packbits((d > np.median(d)).ravel()).view('>u8')[0].astype(np.uint64)

def unit_vec(g, size):
    s = cv2.resize(g.astype(np.float32), (size, size), interpolation=cv2.INTER_AREA).ravel()
    s = s - s.mean()
    return (s / (np.linalg.norm(s) + 1e-8)).astype(np.float16)

def dog_vec(g, size):
    """Band-pass (difference-of-Gaussians) thumbnail: removes the head outline and global intensity so that the
    correlation is driven by internal anatomy (ventricles, sulci, lesion borders), which differs between slices."""
    t = cv2.resize(g.astype(np.float32), (2 * size, 2 * size), interpolation=cv2.INTER_AREA)
    d = cv2.GaussianBlur(t, (0, 0), 1.0) - cv2.GaussianBlur(t, (0, 0), 4.0)
    return unit_vec(d, size)

def descriptor_from_array(a):
    c = brain_crop(a); cf = np.ascontiguousarray(c[:, ::-1]); n = CFG['desc_size']
    return phash64(c), phash64(cf), unit_vec(c, n), unit_vec(cf, n), dog_vec(c, n), dog_vec(cf, n)

def describe_path(args):
    path, flair = args
    try:
        return descriptor_from_array(read_gray(path, flair))
    except Exception:
        return None

def pmap(fn, items, desc, chunksize=16):
    if CFG['n_proc'] <= 1 or len(items) < 64:
        return [fn(x) for x in tqdm(items, desc=desc)]
    with ProcessPoolExecutor(max_workers=CFG['n_proc'], mp_context=mp.get_context('fork')) as ex:
        return list(tqdm(ex.map(fn, items, chunksize=chunksize), total=len(items), desc=desc))

def is_dup(raw, bp, thr=None):
    return (raw >= CFG['dup_ncc']) & (bp >= (CFG['dup_bp'] if thr is None else thr))

def dense_links(DA, DB=None, raw_min=0.80, block=1024):
    """EVERY pair (i, j) whose raw-thumbnail NCC (max over as-is / mirrored) is >= raw_min, found by blockwise
    matrix products (no hash pre-filter, so re-cropped or zoomed copies are not lost at a candidate stage).
    Returns I, J, raw NCC, band-pass NCC (in the orientation chosen by the raw NCC) and the flip-aware pHash distance.
    DB=None: pairs i < j within DA."""
    same = DB is None
    DB = DA if same else DB
    VB, VFB = DB['V'].astype(np.float32), DB['VF'].astype(np.float32)
    BB, BFB = DB['B'].astype(np.float32), DB['BF'].astype(np.float32)
    I, J, R, P = [], [], [], []
    nA = len(DA['V'])
    for s_ in range(0, nA, block):
        e_ = min(nA, s_ + block); c0 = s_ if same else 0          # same set: only columns j >= s_ (upper triangle)
        va = DA['V'][s_:e_].astype(np.float32)
        r0 = va @ VB[c0:].T; r1 = va @ VFB[c0:].T
        raw = np.maximum(r0, r1)
        if same:                                                 # drop j <= i inside the diagonal block
            raw[np.tril_indices(e_ - s_, 0, raw.shape[1])] = -1
        ii, jj = np.nonzero(raw >= raw_min)
        if not len(ii): continue
        fl = r1[ii, jj] > r0[ii, jj]
        ba = DA['B'][s_:e_][ii].astype(np.float32)
        bp = np.where(fl, (ba * BFB[c0 + jj]).sum(1), (ba * BB[c0 + jj]).sum(1))
        I.append(ii + s_); J.append(jj + c0); R.append(raw[ii, jj]); P.append(bp)
    if not I:
        z = np.zeros(0, np.int64); return z, z, np.zeros(0, np.float32), np.zeros(0, np.float32), np.zeros(0, np.int16)
    I, J, R, P = np.concatenate(I), np.concatenate(J), np.concatenate(R).astype(np.float32), np.concatenate(P).astype(np.float32)
    HA, HFA, HB, HFB = DA['H'], DA['HF'], DB['H'], DB['HF']
    ham = np.minimum(np.minimum(popcount64(HA[I] ^ HB[J]), popcount64(HA[I] ^ HFB[J])), popcount64(HFA[I] ^ HB[J]))
    return I, J, R, P, ham.astype(np.int16)

class UnionFind:
    def __init__(self, n): self.p = np.arange(n)
    def find(self, x):
        r = x
        while self.p[r] != r: r = self.p[r]
        while self.p[x] != r: self.p[x], x = r, self.p[x]
        return r
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[max(ra, rb)] = min(ra, rb)

In [ ]:
# ── 6. Duplicate / overlap screening across all sources ────────────────────────────────────────────
SCREEN = ['nick_train', 'nick_test', 'brisc_train', 'brisc_test', 'pmram', 'lgg']
EVAL_SOURCES = [s for s in ['nick_test', 'brisc_test', 'pmram'] if (MAN.source == s).any()]
TRAIN_SOURCES = ['nick_train', 'brisc_train', 'lgg']

scr_path = DIRS['manifests'] / 'screen_descriptors.npz'
SCR = MAN[MAN.source.isin(SCREEN)].reset_index(drop=True)
if scr_path.exists() and json.load(open(DIRS['manifests'] / 'screen_uids.json')) == SCR.uid.tolist():
    z = np.load(scr_path); DESC = {k: z[k] for k in ['H', 'HF', 'V', 'VF', 'B', 'BF']}
    print('✓ descriptors loaded from cache')
else:
    desc = pmap(describe_path, list(zip(SCR.path, SCR.source.eq('lgg'))), 'duplicate descriptors')
    bad = [i for i, d in enumerate(desc) if d is None]
    assert not bad, f'{len(bad)} unreadable images, e.g. {SCR.path[bad[0]]}'
    DESC = dict(H=np.array([d[0] for d in desc], np.uint64), HF=np.array([d[1] for d in desc], np.uint64),
                V=np.stack([d[2] for d in desc]), VF=np.stack([d[3] for d in desc]),
                B=np.stack([d[4] for d in desc]), BF=np.stack([d[5] for d in desc]))
    np.savez(scr_path, **DESC)
    json.dump(SCR.uid.tolist(), open(DIRS['manifests'] / 'screen_uids.json', 'w'))

t_ = time.time()
I, J, RAW, BP, D = dense_links(DESC, raw_min=CFG['pair_raw_min'])
while len(I) > 30_000_000 and CFG['pair_raw_min'] < CFG['dup_ncc']:   # memory guard (not expected)
    CFG['pair_raw_min'] = round(CFG['pair_raw_min'] + 0.02, 2)
    print(f"⚠ {len(I):,} pairs — raising the analysis floor to raw NCC ≥ {CFG['pair_raw_min']}")
    I, J, RAW, BP, D = dense_links(DESC, raw_min=CFG['pair_raw_min'])
print(f"all {len(SCR) * (len(SCR) - 1) // 2:,} image pairs compared in {time.time() - t_:.0f}s | "
      f"pairs with raw NCC ≥ {CFG['pair_raw_min']}: {len(I):,}")
PAIRS = pd.DataFrame(dict(i=I, j=J, hamming=D, ncc=RAW, bp=BP))
_cats = sorted(SCR.source.unique())
PAIRS['src_i'] = pd.Categorical(SCR.source.values[I], categories=_cats)
PAIRS['src_j'] = pd.Categorical(SCR.source.values[J], categories=_cats)
PAIRS['lab_i'] = SCR.label.values[I].astype(np.int8); PAIRS['lab_j'] = SCR.label.values[J].astype(np.int8)
PAIRS['dup'] = is_dup(PAIRS.ncc.values, PAIRS.bp.values)
CONF = PAIRS[PAIRS.dup].reset_index(drop=True)
CONF['uid_i'], CONF['uid_j'] = SCR.uid.values[CONF.i.values], SCR.uid.values[CONF.j.values]
CONF.to_csv(DIRS['results'] / 'near_duplicate_pairs.csv', index=False)
print(f'related / near-duplicate pairs (raw NCC ≥ {CFG["dup_ncc"]} and band-pass NCC ≥ {CFG["dup_bp"]}): {len(CONF):,} | '
      f'of these exact copies (band-pass ≥ {CFG["dup_exact_bp"]}): {int((CONF.bp >= CFG["dup_exact_bp"]).sum()):,}')
# sanity check of link precision: linked T1 images carrying DIFFERENT labels (LGG excluded: glioma-only source)
_nl = CONF[(CONF.src_i != 'lgg') & (CONF.src_j != 'lgg')]
REL_MISMATCH = _nl[_nl.lab_i != _nl.lab_j]
print(f'linked pairs with different labels (possible false links or label errors): {len(REL_MISMATCH):,} of {len(_nl):,}')

uf = UnionFind(len(SCR))
for a, b in zip(CONF.i.values, CONF.j.values): uf.union(int(a), int(b))
SCR['cluster'] = [f'c{uf.find(i)}' for i in range(len(SCR))]
csize = SCR.cluster.value_counts()
print(f'near-duplicate clusters: {int((csize > 1).sum()):,} with ≥2 images; largest has {int(csize.max())} images')
if csize.max() > 50:
    print('⚠ a very large cluster suggests false-positive links — inspect figures/fig_overlap_examples.pdf; '
          'removal decisions below use DIRECT pairs only, so chaining cannot remove unrelated images.')

def overlap_counts(conf):
    """n images of source A that have ≥1 confirmed near-duplicate in source B (direct pairs, both directions)."""
    a = np.concatenate([conf.i.values, conf.j.values]); b = np.concatenate([conf.j.values, conf.i.values])
    sa, sb = SCR.source.values[a], SCR.source.values[b]
    out = {}
    for A in SCREEN:
        for B in SCREEN:
            out[(A, B)] = int(np.unique(a[(sa == A) & (sb == B)]).size)
        tb = np.isin(sb, TRAIN_SOURCES)
        out[(A, 'any_train')] = int(np.unique(a[(sa == A) & tb]).size)
    return out

OVL = {thr: overlap_counts(PAIRS[is_dup(PAIRS.ncc.values, PAIRS.bp.values, thr)])
       for thr in sorted(set(CFG['dup_bp_grid']) | {CFG['dup_bp']})}
N_SRC = SCR.source.value_counts().to_dict()
ov_rows = []
for A in SCREEN:
    for B in SCREEN + ['any_train']:
        r = dict(source=A, n=N_SRC.get(A, 0), matched_in=B)
        for thr, o in OVL.items(): r[f'n_bp{thr:.2f}'] = o[(A, B)]
        ov_rows.append(r)
OVERLAP = pd.DataFrame(ov_rows); OVERLAP.to_csv(DIRS['results'] / 'overlap_summary.csv', index=False)
LABEL_CONFLICTS = CONF[(CONF.lab_i != CONF.lab_j) & (CONF.bp >= CFG['dup_exact_bp'])]   # identical image, different label
LABEL_CONFLICTS.to_csv(DIRS['results'] / 'label_conflicts.csv', index=False)
print(OVERLAP[OVERLAP.source.isin(EVAL_SOURCES) & OVERLAP.matched_in.isin(TRAIN_SOURCES + ['any_train'])]
      .to_string(index=False))
print('label conflicts among exact copies:', len(LABEL_CONFLICTS))

duplicate descriptors:   0%|          | 0/18634 [00:00<?, ?it/s]

all 173,603,661 image pairs compared in 4s | pairs with raw NCC ≥ 0.8: 88,269
related / near-duplicate pairs (raw NCC ≥ 0.85 and band-pass NCC ≥ 0.85): 32,728 | of these exact copies (band-pass ≥ 0.92): 23,756
linked pairs with different labels (possible false links or label errors): 45 of 32,135
near-duplicate clusters: 3,318 with ≥2 images; largest has 48 images
    source    n  matched_in  n_bp0.80  n_bp0.85  n_bp0.92  n_bp0.97
 nick_test 1600  nick_train      1189      1135      1040      1009
 nick_test 1600 brisc_train       823       768       677       665
 nick_test 1600         lgg         1         0         0         0
 nick_test 1600   any_train      1230      1179      1089      1057
brisc_test 1000  nick_train       666       629       553       544
brisc_test 1000 brisc_train       659       623       540       531
brisc_test 1000         lgg         1         0         0         0
brisc_test 1000   any_train       688       653       575       564
     pmram 1505  nick

In [ ]:
# ── 7. Patient IDs for figshare-derived images (Cheng et al. .mat files → PID) ──────────────────────
FS_LABEL = {1: 1, 2: 0, 3: 2}    # figshare: 1 meningioma, 2 glioma, 3 pituitary

def _fig_desc(mp_):
    try:
        import h5py
        with h5py.File(mp_, 'r') as f:
            if 'cjdata' not in f: return None
            cj = f['cjdata']
            img = np.array(cj['image'], dtype=np.float32)
            lab = int(np.array(cj['label']).ravel()[0])
            pid = ''.join(chr(int(c)) for c in np.array(cj['PID']).ravel()).strip()
    except Exception:
        try:
            from scipy.io import loadmat
            m = loadmat(mp_, simplify_cells=True)['cjdata']
            img, lab, pid = np.asarray(m['image'], np.float32), int(m['label']), str(m['PID']).strip()
        except Exception:
            return None
    # h5py returns MATLAB arrays transposed; both orientations are described and the better match is kept
    return (str(mp_), lab, pid, descriptor_from_array(img.T), descriptor_from_array(img))

FIG_MAP = pd.DataFrame(columns=['uid', 'fs_pid', 'fs_label', 'fs_ncc'])
if P_FIG is not None:
    mats = sorted(P_FIG.rglob('*.mat'))
    if not mats:                                    # some mirrors keep the original zip parts
        for zp in sorted(P_FIG.rglob('*.zip')):
            with zipfile.ZipFile(zp) as zf: zf.extractall(LOCAL / 'figshare')
        mats = sorted((LOCAL / 'figshare').rglob('*.mat'))
    if SMOKE_TEST: mats = mats[:300]
    fd = [d for d in pmap(_fig_desc, mats, 'figshare descriptors') if d is not None]
    print(f'figshare .mat files read: {len(fd)} | patients: {len({d[2] for d in fd})}')
    if fd:
        FD = dict(H=np.array([[d[3][0], d[4][0]] for d in fd], np.uint64).ravel(),       # 2 orientations per file
                  HF=np.array([[d[3][1], d[4][1]] for d in fd], np.uint64).ravel(),
                  V=np.stack([v for d in fd for v in (d[3][2], d[4][2])]), VF=np.stack([v for d in fd for v in (d[3][3], d[4][3])]),
                  B=np.stack([v for d in fd for v in (d[3][4], d[4][4])]), BF=np.stack([v for d in fd for v in (d[3][5], d[4][5])]))
        tgt = np.flatnonzero(SCR.source.isin(['nick_train', 'nick_test', 'brisc_train', 'brisc_test']).values)
        DT = {k: v[tgt] for k, v in DESC.items()}
        fi, tj, raw_, bp_, _ = dense_links(DT, FD, raw_min=CFG['dup_ncc'])
        if len(fi):
            M = pd.DataFrame(dict(t=tgt[fi], f=tj // 2, ncc=raw_, bp=bp_))
            # exact copies only (an adjacent slice would also give the right patient, but a copy is unambiguous)
            M = M[is_dup(M.ncc.values, M.bp.values, CFG['dup_exact_bp'])].sort_values('bp', ascending=False).drop_duplicates('t')
            FIG_MAP = pd.DataFrame(dict(uid=SCR.uid.values[M.t.values],
                                        fs_pid=['FS:' + fd[k][2] for k in M.f.values],
                                        fs_label=[FS_LABEL.get(fd[k][1], -1) for k in M.f.values],
                                        fs_ncc=M.bp.values))
    FIG_MAP.to_csv(DIRS['results'] / 'figshare_pid_map.csv', index=False)
SCR = SCR.merge(FIG_MAP[['uid', 'fs_pid', 'fs_label']], on='uid', how='left')
SCR['pid'] = np.where(SCR.fs_pid.notna(), SCR.fs_pid, SCR.pid)
lab_ok = (SCR.fs_label.isna() | (SCR.fs_label == SCR.label)).mean()
print(f'figshare-traced images: {SCR.fs_pid.notna().sum():,} | label agreement with figshare: {lab_ok:.3f}')
print(SCR[SCR.fs_pid.notna()].groupby('source').agg(n=('uid', 'size'), patients=('pid', 'nunique')))

figshare descriptors:   0%|          | 0/3065 [00:00<?, ?it/s]

figshare .mat files read: 3064 | patients: 233
figshare-traced images: 9,025 | label agreement with figshare: 0.999
                n  patients
source                     
brisc_test    700       204
brisc_train  3653       228
nick_test     857       209
nick_train   3815       232


In [ ]:
# ── 8. Partition: decontaminate the training pool, grouped validation split, fixed test sets ──────
PRIORITY = {'nick_train': 0, 'brisc_train': 1}
TEST_ROLE = {'nick_test': 'test_nick', 'brisc_test': 'test_brisc', 'pmram': 'test_pmram'}

def build_partition(scr, pairs):
    """Decisions use DIRECT pairs only (no transitive chaining). 'Related' links (copies, re-crops, adjacent slices;
    pairs.dup) decide what is removed because of the test sets and how validation is grouped; only EXACT copies are
    de-duplicated inside the training pool, so different slices of one scan stay available for training."""
    P = scr.copy(); P['role'] = ''
    for s_, r_ in TEST_ROLE.items(): P.loc[P.source == s_, 'role'] = r_
    conf = pairs[pairs.dup]
    a = np.concatenate([conf.i.values, conf.j.values]); b = np.concatenate([conf.j.values, conf.i.values])
    src = P.source.values
    is_eval, is_cand = np.isin(src, EVAL_SOURCES), np.isin(src, ['nick_train', 'brisc_train', 'lgg'])
    # (1) any training candidate with a near-duplicate among the test images is removed
    P.loc[np.unique(a[is_cand[a] & is_eval[b]]), 'role'] = 'excluded_dup_of_test'
    # (2) greedy de-duplication of EXACT copies in the T1 pool (Nickparvar copy kept first);
    #     exact copies with conflicting labels → both removed
    ex = pairs[is_dup(pairs.ncc.values, pairs.bp.values, CFG['dup_exact_bp'])]
    ae = np.concatenate([ex.i.values, ex.j.values]); be = np.concatenate([ex.j.values, ex.i.values])
    pool = set(P.index[P.source.isin(['nick_train', 'brisc_train']) & P.role.eq('')])
    nbrs, conflict = defaultdict(set), set()
    for x, y in zip(ae, be):
        if x in pool and y in pool:
            nbrs[x].add(y)
            if P.label[x] != P.label[y]: conflict.update((x, y))
    kept = set()
    for i in sorted(pool, key=lambda k: (PRIORITY[P.source[k]], P.uid[k])):
        if i in conflict: P.loc[i, 'role'] = 'excluded_label_conflict'
        elif nbrs[i] & kept: P.loc[i, 'role'] = 'excluded_dup_within_train'
        else: kept.add(i)
    # (3) LGG: tumour-bearing slices → mask-guided crops for training; tumour-free slices → shortcut probe only
    lgg = P.source.eq('lgg') & P.role.eq('')
    P.loc[lgg & P.lgg_tumor, 'role'] = 'train_lgg_slice'
    P.loc[lgg & ~P.lgg_tumor, 'role'] = 'probe_lgg_notumour'
    # (4) validation split from the kept T1 pool: stratified by class, grouped by patient (figshare PID) and by
    #     'related' links (re-crops, adjacent slices), so related images never straddle train/validation
    kept = sorted(kept); loose = pairs[pairs.dup]
    uf_ = UnionFind(len(P)); ks = set(kept)
    for x, y in zip(loose.i.values, loose.j.values):
        if x in ks and y in ks: uf_.union(int(x), int(y))
    pid = P.pid.fillna(''); first = {}
    for i in kept:                                   # images of one (figshare) patient form one group as well
        if pid[i]:
            if pid[i] in first: uf_.union(first[pid[i]], int(i))
            else: first[pid[i]] = int(i)
    groups = np.array([f'g{uf_.find(i)}' for i in kept])
    gs = pd.Series(groups).value_counts()
    print(f'validation grouping: {len(gs):,} groups for {len(kept):,} images; largest group {int(gs.iloc[0]):,} images')
    if gs.iloc[0] > 0.05 * len(kept):
        print('⚠ one group holds >5% of the pool — inspect near_duplicate_pairs.csv for chained false links')
    n_splits = max(2, int(round(1 / CFG['val_frac'])))
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=CFG['split_seed'])
    _, val_pos = next(sgkf.split(np.zeros(len(kept)), P.label.values[kept], groups))
    P.loc[kept, 'role'] = 'train'
    P.loc[np.array(kept)[val_pos], 'role'] = 'val'
    P['group'] = ''; P.loc[kept, 'group'] = groups
    assert not P.role.eq('').any()
    return P

PART = build_partition(SCR, PAIRS)
TEST_PIDS = set(PART.loc[PART.role.str.startswith('test') & PART.pid.fillna('').ne(''), 'pid'])
# sanity checks: no direct near-duplicate and no known patient spans train/val and test; train vs val grouped
_c = PAIRS[PAIRS.dup]
_tr = PART.role.isin(['train', 'val', 'train_lgg_slice']).to_numpy(); _te = PART.role.str.startswith('test').to_numpy()
assert not ((_tr[_c.i] & _te[_c.j]) | (_te[_c.i] & _tr[_c.j])).any(), 'near-duplicate between training and test'
_g = PART.loc[PART.role.isin(['train', 'val'])]
assert not set(_g.loc[_g.role == 'train', 'group']) & set(_g.loc[_g.role == 'val', 'group']), 'group spans train/val'
_trn = PART.role.eq('train').to_numpy(); _val = PART.role.eq('val').to_numpy()
assert not ((_trn[_c.i] & _val[_c.j]) | (_val[_c.i] & _trn[_c.j])).any(), 'related images span train/val'
PART.to_csv(DIRS['manifests'] / 'partition_manifest.csv', index=False)   # supplementary data (source, PID, cluster, role)
print(PART.groupby(['role', 'source']).size().unstack(fill_value=0))
print(f"test-set patients (figshare PIDs): {len(TEST_PIDS)} | of these also in training (before patient exclusion): "
      f"{len(TEST_PIDS & set(PART.loc[PART.role.isin(['train', 'val']), 'pid']))}")

validation grouping: 1,521 groups for 2,909 images; largest group 31 images
source                     brisc_test  brisc_train   lgg  nick_test  \
role                                                                  
excluded_dup_of_test                0         1834     0          0   
excluded_dup_within_train           0         2205     0          0   
excluded_label_conflict             0            6     0          0   
probe_lgg_notumour                  0            0  2556          0   
test_brisc                       1000            0     0          0   
test_nick                           0            0     0       1600   
test_pmram                          0            0     0          0   
train                               0          836     0          0   
train_lgg_slice                     0            0  1373          0   
val                                 0          119     0          0   

source                     nick_train  pmram  
role                    

In [ ]:
# ── 9. Intensity normalisation (White Stripe with a fixed window; percentile variant for the ablation) ──
PRE = LOCAL / 'pre'
USED_ROLES = ['train', 'val', 'test_nick', 'test_brisc', 'test_pmram', 'train_lgg_slice', 'probe_lgg_notumour']
U = PART[PART.role.isin(USED_ROLES)].copy()

def _ws_pass1(args):
    path, flair, contrast, want_z, seed = args
    try:
        a = read_gray(path, flair)
        st = whitestripe_stats(a, contrast, seed=seed)
        zs = None
        if st is not None and want_z:
            m = foreground_mask(a); v = a[m]
            rng = np.random.default_rng(seed)
            v = rng.choice(v, min(2000, v.size), replace=False)
            zs = ((v - st['mu']) / st['sd']).astype(np.float32)
        return st, zs
    except Exception:
        return None, None

def _seed_of(uid): return int(hashlib.sha1(uid.encode()).hexdigest()[:8], 16)
def _out_path(variant, source, uid, suffix=''):
    return PRE / variant / source / f'{hashlib.sha1(uid.encode()).hexdigest()[:20]}{suffix}.png'

pass1 = pmap(_ws_pass1, [(r.path, r.source == 'lgg', r.contrast, r.role in ('train', 'train_lgg_slice'),
                           _seed_of(r.uid)) for r in U.itertuples()], 'White Stripe stats')
U['ws_ok'] = [p[0] is not None for p in pass1]
U['ws_mu'] = [p[0]['mu'] if p[0] else np.nan for p in pass1]
U['ws_sd'] = [p[0]['sd'] if p[0] else np.nan for p in pass1]
WINDOW = {}
for c in ['T1', 'T2']:
    zs = [p[1] for p, con in zip(pass1, U.contrast) if con == c and p[1] is not None]
    WINDOW[c] = tuple(float(x) for x in np.percentile(np.concatenate(zs), CFG['ws_window_pct'])) if zs else (-10., 10.)
print('White Stripe windows (z):', WINDOW, '| fallback rate:', f"{1 - U.ws_ok.mean():.3%}")

def lgg_crops(img, mask, n, seed):
    rng = random.Random(seed)
    ys, xs = np.nonzero(mask)
    x, y, w, h = xs.min(), ys.min(), xs.max() - xs.min() + 1, ys.max() - ys.min() + 1
    cx, cy = x + w // 2, y + h // 2
    H_, W_ = img.shape[:2]; out = []
    for _ in range(n):
        j = rng.randint(-CFG['lgg_jitter'], CFG['lgg_jitter'])
        half = min(max(w, h) // 2 + CFG['lgg_margin'] + abs(j), min(H_, W_) // 2)
        x1, y1 = max(0, cx - half + j), max(0, cy - half + j)
        x2, y2 = min(W_, cx + half + j), min(H_, cy + half + j)
        crop = img[y1:y2, x1:x2]
        if crop.shape[0] < 32 or crop.shape[1] < 32: continue
        out.append(cv2.resize(crop, (CFG['img_size'], CFG['img_size']), interpolation=cv2.INTER_LANCZOS4))
    return out

def _ws_pass2(args):
    uid, path, source, contrast, role, mask_path, mu, sd, ok = args
    try:
        a = read_gray(path, source == 'lgg')
        imgs = {'ws': ws_to_u8(a, dict(mu=mu, sd=sd), WINDOW[contrast]) if ok else percentile_u8(a),
                'percentile': percentile_u8(a)}
        res = {}
        for var, im in imgs.items():
            if role == 'train_lgg_slice':
                mask = np.array(Image.open(mask_path)) > 0
                paths = []
                for k, cr in enumerate(lgg_crops(im, mask, CFG['lgg_crops_per'], _seed_of(uid))):
                    op = _out_path(var, 'lgg_crops', uid, f'_c{k}'); op.parent.mkdir(parents=True, exist_ok=True)
                    Image.fromarray(cr).save(op); paths.append(str(op))
                res[var] = paths
            else:
                op = _out_path(var, source, uid); op.parent.mkdir(parents=True, exist_ok=True)
                Image.fromarray(im).save(op); res[var] = str(op)
        return res
    except Exception as e:
        return {'error': f'{uid}: {e}'}

pass2 = pmap(_ws_pass2, [(r.uid, r.path, r.source, r.contrast, r.role, r.mask_path, r.ws_mu, r.ws_sd, r.ws_ok)
                         for r in U.itertuples()], 'normalise + write')
errs = [p['error'] for p in pass2 if 'error' in p]
assert not errs, errs[:5]
for var in ['ws', 'percentile']:
    U[f'pre_{var}'] = [p[var] for p in pass2]
LGG_CROPS = {var: [(cp, 0, pid) for paths, pid in zip(U.loc[U.role == 'train_lgg_slice', f'pre_{var}'],
                                                        U.loc[U.role == 'train_lgg_slice', 'pid']) for cp in paths]
             for var in ['ws', 'percentile']}
U.to_pickle(DIRS['manifests'] / 'used_images.pkl')
save_json(dict(window=WINDOW, fallback_rate=float(1 - U.ws_ok.mean()),
               fallback_rate_T1=float(1 - U.loc[U.contrast == 'T1', 'ws_ok'].mean()),
               fallback_rate_T2=float(1 - U.loc[U.contrast == 'T2', 'ws_ok'].mean()),
               n_lgg_crops=len(LGG_CROPS['ws'])), DIRS['results'] / 'preprocessing.json')
print(f"✓ normalised images written | LGG crops: {len(LGG_CROPS['ws'])} from "
      f"{(U.role == 'train_lgg_slice').sum()} tumour-bearing slices of {U.loc[U.role == 'train_lgg_slice', 'pid'].nunique()} patients")

White Stripe stats:   0%|          | 0/10943 [00:00<?, ?it/s]

White Stripe windows (z): {'T1': (-57.56638717651367, 91.85000610351562), 'T2': (-64.99233261108398, 96.38742065429688)} | fallback rate: 1.745%


normalise + write:   0%|          | 0/10943 [00:00<?, ?it/s]

✓ normalised images written | LGG crops: 4119 from 1373 tumour-bearing slices of 110 patients


In [ ]:
# ── 10. PyTorch setup: transforms, datasets, model and loss (unchanged from v5) ─────────────────────
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.swa_utils import AveragedModel, SWALR
import torchvision.transforms as T
import timm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'Select a GPU runtime (Runtime → Change runtime type).'
GPU_NAME = torch.cuda.get_device_name(0)
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
USE_SCALER = AMP_DTYPE == torch.float16
EVAL_WORKERS = max(2, min(8, os.cpu_count() or 2))
print(f'✓ {GPU_NAME} | {torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB | AMP {AMP_DTYPE} | '
      f'torch {torch.__version__} | timm {timm.__version__}')

IMG_MEAN, IMG_STD, SZ = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225], CFG['img_size']
train_transform = T.Compose([
    T.Resize((SZ + 32, SZ + 32)), T.RandomCrop(SZ),
    T.RandomHorizontalFlip(0.5), T.RandomVerticalFlip(0.3), T.RandomRotation(20),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1), shear=5),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.1),
    T.RandomGrayscale(0.1), T.GaussianBlur(3, sigma=(0.1, 2.0)),
    T.ToTensor(), T.Normalize(IMG_MEAN, IMG_STD),
    T.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])
_norm = [T.ToTensor(), T.Normalize(IMG_MEAN, IMG_STD)]
val_transform = T.Compose([T.Resize((SZ, SZ))] + _norm)
TTA_VIEWS = [
    T.Compose([T.Resize((SZ, SZ))] + _norm),                                                   # 1 identity
    T.Compose([T.Resize((SZ, SZ)), T.RandomHorizontalFlip(p=1.0)] + _norm),                    # 2 h-flip
    T.Compose([T.Resize((SZ, SZ)), T.RandomVerticalFlip(p=1.0)] + _norm),                      # 3 v-flip
    T.Compose([T.Resize((SZ + 64, SZ + 64)), T.CenterCrop(SZ)] + _norm),                       # 4 centre crop
    T.Compose([T.Resize((SZ, SZ)), T.RandomRotation((90, 90))] + _norm),                       # 5 rot 90
    T.Compose([T.Resize((SZ, SZ)), T.RandomRotation((180, 180))] + _norm),                     # 6 rot 180
    T.Compose([T.Resize((SZ, SZ)), T.RandomRotation((270, 270))] + _norm),                     # 7 rot 270
    T.Compose([T.Resize((SZ, SZ)), T.RandomRotation((-15, 15))] + _norm),                      # 8 random ±15°
    T.Compose([T.Resize((SZ + 32, SZ + 32)), T.RandomCrop(SZ)] + _norm),                       # 9 random crop
    T.Compose([T.Resize((SZ, SZ)), T.ColorJitter(brightness=0.2)] + _norm),                    # 10 brightness
    T.Compose([T.Resize((SZ, SZ)), T.ColorJitter(contrast=0.2)] + _norm),                      # 11 contrast
    T.Compose([T.Resize((int(SZ * 0.9), int(SZ * 0.9))), T.Pad(SZ // 20), T.Resize((SZ, SZ))] + _norm),  # 12 zoom-out
    T.Compose([T.Resize((SZ, SZ)), T.RandomHorizontalFlip(p=1.0), T.RandomRotation((90, 90))] + _norm),  # 13
    T.Compose([T.Resize((SZ, SZ)), T.GaussianBlur(3, sigma=1.0)] + _norm),                     # 14 blur
]
assert len(TTA_VIEWS) == CFG['tta_k']

class ImgDS(Dataset):
    def __init__(self, items, tfm): self.items, self.tfm = items, tfm
    def __len__(self): return len(self.items)
    def __getitem__(self, i):
        it = self.items[i]
        return self.tfm(Image.open(it[0]).convert('RGB')), int(it[1])

class BrainTumorViT(nn.Module):
    """ViT-Base/16@384 + MLP head 768→512→256→4 (exactly as trained in v5)."""
    def __init__(self, num_classes=4, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(CFG['backbone'], pretrained=pretrained, num_classes=0,
                                          drop_rate=CFG['drop_rate'], img_size=CFG['img_size'])
        feat = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(feat), nn.Linear(feat, 512), nn.GELU(), nn.Dropout(0.3),
            nn.LayerNorm(512), nn.Linear(512, 256), nn.GELU(), nn.Dropout(0.15),
            nn.LayerNorm(256), nn.Linear(256, num_classes), nn.Dropout(0.1))
        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
    def forward(self, x): return self.head(self.backbone(x))
    def get_param_groups(self):
        blocks = list(self.backbone.blocks); t = len(blocks) // 4
        return [{'params': self.head.parameters(), 'lr': CFG['lr_head']},
                {'params': [p for b in blocks[-t:] for p in b.parameters()], 'lr': CFG['lr_backbone'] * 2},
                {'params': [p for b in blocks[-2 * t:-t] for p in b.parameters()], 'lr': CFG['lr_backbone']},
                {'params': list(self.backbone.patch_embed.parameters()) +
                           [p for b in blocks[:-2 * t] for p in b.parameters()], 'lr': CFG['lr_backbone'] * 0.3}]

def fresh_model(pretrained=True): return BrainTumorViT(pretrained=pretrained).to(DEVICE)

class AsymmetricFocalLoss(nn.Module):
    """Numerically stable ASL (fp32 loss math, fused logsigmoid, clamped focal weight, detached focal weight)."""
    def __init__(self, gamma_pos=0.0, gamma_neg=4.0, clip=0.05, eps=1e-5):
        super().__init__(); self.gp, self.gn, self.clip, self.eps = gamma_pos, gamma_neg, clip, eps
    def forward(self, logits, targets):
        with torch.amp.autocast('cuda', enabled=False):
            logits = logits.float(); oht = F.one_hot(targets, logits.shape[-1]).float()
            los_pos = oht * F.logsigmoid(logits)
            xs_neg = (1 - torch.sigmoid(logits) + self.clip).clamp(max=1)
            los_neg = (1 - oht) * torch.log(xs_neg.clamp(min=self.eps))
            loss = los_pos + los_neg
            if self.gp > 0 or self.gn > 0:
                xs_pos = torch.sigmoid(logits)
                pt = xs_pos * oht + (1 - xs_pos) * (1 - oht)
                gamma = self.gp * oht + self.gn * (1 - oht)
                loss = loss * (1 - pt).clamp(min=self.eps).pow(gamma).detach()
            return -loss.mean()

def train_epoch(model, loader, optimizer, criterion, scheduler=None, scaler=None):
    model.train(); loss_sum = correct = total = nan_sk = 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
            logits = model(imgs)
        loss = criterion(logits, labels)
        if not torch.isfinite(loss): nan_sk += 1; continue
        if scaler is not None:
            scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['max_norm'])
            scaler.step(optimizer); scaler.update()
        else:
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['max_norm']); optimizer.step()
        if scheduler is not None: scheduler.step()
        loss_sum += loss.item() * imgs.size(0); correct += (logits.argmax(1) == labels).sum().item(); total += imgs.size(0)
    if nan_sk: print(f'    ⚠ {nan_sk} non-finite batches skipped')
    return loss_sum / max(total, 1), 100 * correct / max(total, 1)

@torch.no_grad()
def predict_logits(model, items, tfm, seed=None, bs=None):
    """Raw logits (N, C). `seed` makes the stochastic TTA views reproducible (worker seeds derive from it)."""
    if seed is not None: torch.manual_seed(seed); random.seed(seed); np.random.seed(seed)
    ld = DataLoader(ImgDS(items, tfm), batch_size=bs or CFG['eval_batch_size'], shuffle=False,
                    num_workers=EVAL_WORKERS, pin_memory=True)
    model.eval(); out = []
    for x, _ in ld:
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE):
            out.append(model(x.to(DEVICE, non_blocking=True)).float().cpu())
    return torch.cat(out).numpy()

def tta_logits(model, items):
    """(N, K, C) logits for the K = 14 views; views 8–11 are random transforms with a fixed seed per view."""
    return np.stack([predict_logits(model, items, TTA_VIEWS[k], seed=CFG['tta_seed'] + k)
                     for k in range(CFG['tta_k'])], axis=1)

def fit_temperature(logits, labels):
    """Single temperature T > 0 minimising validation NLL (L-BFGS on log T)."""
    z = torch.tensor(logits, dtype=torch.float32); y = torch.tensor(labels, dtype=torch.long)
    logT = torch.tensor([math.log(1.5)], requires_grad=True)
    opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=200, line_search_fn='strong_wolfe')
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(z / logT.exp(), y); loss.backward(); return loss
    opt.step(closure)
    return float(logT.exp().item())

def make_sampler(labels, mode, boost, seed):
    if mode == 'shuffle': return None
    cnt = Counter(labels); w = {c: 1.0 / n for c, n in cnt.items()}
    if 0 in w: w[0] *= boost
    g = torch.Generator(); g.manual_seed(seed)
    return WeightedRandomSampler(torch.tensor([w[y] for y in labels], dtype=torch.double),
                                 num_samples=len(labels), replacement=True, generator=g)

✓ NVIDIA RTX PRO 6000 Blackwell Server Edition | 102 GB | AMP torch.bfloat16 | torch 2.11.0+cu128 | timm 1.0.29


In [ ]:
# ── 11. One experiment = one full training + validation-based selection + calibration + test inference ──
SRC_KEY = {'nick': 'nick_train', 'brisc': 'brisc_train'}
TEST_ROLES = [('test_nick', 'nick'), ('test_brisc', 'brisc'), ('test_pmram', 'pmram'), ('probe_lgg_notumour', 'probe')]

def items_for_run(rc):
    var = rc['norm']
    tr = U[(U.role == 'train') & U.source.isin([SRC_KEY[s] for s in rc['sources'] if s in SRC_KEY])]
    va = U[U.role == 'val']
    n_excl = 0
    if rc['patient_policy'] == 'exclude':
        mtr, mva = tr.pid.fillna('').isin(TEST_PIDS), va.pid.fillna('').isin(TEST_PIDS)
        n_excl = int(mtr.sum() + mva.sum()); tr, va = tr[~mtr], va[~mva]
    train_items = list(zip(tr[f'pre_{var}'], tr.label))
    if 'lgg' in rc['sources']: train_items += [(p, y) for p, y, _ in LGG_CROPS[var]]
    val_items = list(zip(va[f'pre_{var}'], va.label, va.uid))
    return train_items, val_items, n_excl

def test_sets(var):
    out = {}
    for role, name in TEST_ROLES:
        d = U[U.role == role]
        if len(d): out[name] = list(zip(d[f'pre_{var}'], d.label, d.uid))
    return out

def run_fingerprint(rc, train_items, val_items):
    """Identifies the exact training/validation images and settings of a run, so that a finished run is only reused
    when nothing changed (e.g. not after a change to the duplicate screen or the split)."""
    h = hashlib.sha1(json.dumps({k: rc[k] for k in sorted(rc) if k not in ('label', 'enabled')},
                                sort_keys=True, default=str).encode())
    for p_, y_ in sorted(train_items): h.update(f'{Path(p_).name}:{y_};'.encode())
    for p_, y_, u_ in sorted(val_items): h.update(f'{u_}:{y_};'.encode())
    return h.hexdigest()[:16]

def run_experiment(rc):
    rdir = DIRS['runs'] / rc['name']; rdir.mkdir(parents=True, exist_ok=True)
    train_items, val_items, n_excl = items_for_run(rc)
    fp = run_fingerprint(rc, train_items, val_items)
    if (rdir / 'DONE').exists():
        try: old_fp = json.load(open(rdir / 'summary.json')).get('fingerprint')
        except Exception: old_fp = None
        if old_fp == fp:
            print(f"↷ {rc['name']}: finished earlier with the same data — skipping"); return
        print(f"⚠ {rc['name']}: an earlier result was trained on a different split — retraining")
        for f_ in rdir.glob('*'): f_.unlink()
    print('\n' + '═' * 78 + f"\n{rc['name']}: {rc['label']}\n" + '═' * 78)
    t_run = time.time(); seed_everything(rc['seed'])
    labels = [y for _, y in train_items]; val_lab = np.array([y for _, y, _ in val_items])
    print(f'train {len(train_items)} {dict(sorted(Counter(labels).items()))} | val {len(val_items)} | '
          f'excluded (patient policy) {n_excl}')
    sampler = make_sampler(labels, rc['sampler'], rc['boost'], rc['seed'])
    train_ld = DataLoader(ImgDS(train_items, train_transform), batch_size=CFG['batch_size'], sampler=sampler,
                          shuffle=sampler is None, num_workers=CFG['num_workers'], pin_memory=True, drop_last=True,
                          persistent_workers=True, prefetch_factor=2,
                          generator=torch.Generator().manual_seed(rc['seed']))
    model, criterion = fresh_model(), AsymmetricFocalLoss()
    scaler = torch.amp.GradScaler('cuda') if USE_SCALER else None
    hist = []

    def log_epoch(ep, phase, tl, ta, t0):
        vlog = predict_logits(model, val_items, val_transform)
        vloss = float(criterion(torch.tensor(vlog), torch.tensor(val_lab)))
        vacc = float((vlog.argmax(1) == val_lab).mean() * 100)
        hist.append(dict(epoch=ep, phase=phase, tr_loss=tl, tr_acc=ta, val_loss=vloss, val_acc=vacc,
                         sec=time.time() - t0))
        print(f'  ep {ep:02d} {phase:18s} | tr {tl:.4f} {ta:6.2f}% | val {vloss:.4f} {vacc:6.2f}% | {time.time() - t0:.0f}s')
        return vacc

    # Phase 1 — head-only warm-up
    for p in model.backbone.parameters(): p.requires_grad = False
    opt_w = AdamW(model.head.parameters(), lr=CFG['lr_warmup'], weight_decay=CFG['weight_decay'])
    ep = 0
    for _ in range(CFG['warmup_epochs']):
        ep += 1; t0 = time.time()
        tl, ta = train_epoch(model, train_ld, opt_w, criterion, None, scaler)
        log_epoch(ep, 'P1 warm-up', tl, ta, t0)
    # Phase 2 — 4-tier discriminative fine-tuning (OneCycle); Phase 3 — SWA (last swa_anneal_epochs epochs)
    for p in model.backbone.parameters(): p.requires_grad = True
    opt = AdamW(model.get_param_groups(), weight_decay=CFG['weight_decay'])
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=[CFG['lr_head'], CFG['lr_backbone'] * 2, CFG['lr_backbone'], CFG['lr_backbone'] * 0.3],
        total_steps=CFG['finetune_epochs'] * len(train_ld), pct_start=CFG['pct_start'], anneal_strategy='cos')
    swa_model = AveragedModel(model)
    swa_sched = SWALR(opt, swa_lr=CFG['swa_lr'], anneal_epochs=CFG['swa_anneal_epochs'])
    best_acc, best_state, best_epoch = -1.0, None, None
    for fe in range(1, CFG['finetune_epochs'] + 1):
        ep += 1; t0 = time.time()
        if fe <= CFG['swa_start']:
            tl, ta = train_epoch(model, train_ld, opt, criterion, sched, scaler); phase = 'P2 fine-tune'
        else:
            tl, ta = train_epoch(model, train_ld, opt, criterion, None, scaler)
            swa_model.update_parameters(model); swa_sched.step(); phase = 'P3 SWA'
        va = log_epoch(ep, phase, tl, ta, t0)
        if va > best_acc:
            best_acc, best_epoch = va, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    del train_ld; gc.collect()
    train_minutes = (time.time() - t_run) / 60

    # Model selection on the VALIDATION split only (ViT has no BatchNorm → no SWA BN re-estimation needed)
    swa_net = swa_model.module.eval()
    best_net = fresh_model(pretrained=False); best_net.load_state_dict(best_state); best_net.eval()
    vl = {'swa': predict_logits(swa_net, val_items, val_transform),
          'best': predict_logits(best_net, val_items, val_transform)}
    vacc = {k: float((v.argmax(1) == val_lab).mean() * 100) for k, v in vl.items()}
    selected = 'swa' if vacc['swa'] >= vacc['best'] else 'best'
    temps = {k: fit_temperature(v, val_lab) for k, v in vl.items()}
    np.savez_compressed(rdir / 'val_logits.npz', swa=vl['swa'], best=vl['best'], labels=val_lab,
                        uids=np.array([u for _, _, u in val_items]))
    nets = {'swa': swa_net, 'best': best_net}
    to_eval = ['swa', 'best'] if rc['name'] == 'main' else [selected]
    for m in to_eval:
        for tname, items in test_sets(rc['norm']).items():
            t0 = time.time(); L = tta_logits(nets[m], items)
            np.savez_compressed(rdir / f'logits_{m}_{tname}.npz', logits=L.astype(np.float32),
                                labels=np.array([y for _, y, _ in items]), uids=np.array([u for _, _, u in items]))
            print(f'  test {m:4s} {tname:6s} n={len(items):5d} | single-view acc '
                  f'{(L[:, 0].argmax(1) == np.array([y for _, y, _ in items])).mean() * 100:6.2f}% | {time.time() - t0:.0f}s')
    if rc['name'] == 'main':
        torch.save(nets[selected].state_dict(), DIRS['checkpoints'] / 'main_selected.pt')
    summary = dict(rc, fingerprint=fp, selected=selected, best_epoch=best_epoch, val_acc_best=vacc['best'], val_acc_swa=vacc['swa'],
                   T=temps[selected], T_swa=temps['swa'], T_best=temps['best'], n_train=len(train_items),
                   n_train_by_class={CLASSES[k]: v for k, v in sorted(Counter(labels).items())},
                   n_val=len(val_items), n_excluded_patient=n_excl, train_minutes=train_minutes,
                   epoch_seconds=float(np.mean([h['sec'] for h in hist])), total_minutes=(time.time() - t_run) / 60,
                   gpu=GPU_NAME, torch=torch.__version__, timm=timm.__version__)
    save_json(summary, rdir / 'summary.json'); save_json(hist, rdir / 'history.json')
    (rdir / 'DONE').write_text('ok')
    print(f"✓ {rc['name']} done in {summary['total_minutes']:.1f} min | selected={selected} | T*={temps[selected]:.4f}")
    del model, swa_model, swa_net, best_net, nets; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ── 12. Train everything in RUN_PLAN (resumes automatically after a disconnect) ────────────────────
for rc in RUN_PLAN:
    if rc.get('enabled', True):
        run_experiment(rc)
print('✓ all planned runs finished')


══════════════════════════════════════════════════════════════════════════════
main: Full model (BTFuseViT)
══════════════════════════════════════════════════════════════════════════════
train 6697 {0: 4906, 1: 494, 2: 287, 3: 1010} | val 331 | excluded (patient policy) 0


model.safetensors: reconstructing file:   0%|          |  0.00B /  347MB            

model.safetensors: downloading bytes:           |  0.00B            

  ep 01 P1 warm-up         | tr 0.2157  29.42% | val 0.2640  24.77% | 12s
  ep 02 P1 warm-up         | tr 0.2253  40.17% | val 0.2700  44.11% | 10s
  ep 03 P1 warm-up         | tr 0.2188  46.22% | val 0.2676  44.11% | 10s
  ep 04 P1 warm-up         | tr 0.2091  51.11% | val 0.2568  47.73% | 10s
  ep 05 P1 warm-up         | tr 0.1992  56.51% | val 0.2529  50.15% | 10s
  ep 06 P2 fine-tune       | tr 0.1581  70.03% | val 0.1799  71.00% | 22s
  ep 07 P2 fine-tune       | tr 0.0830  88.16% | val 0.1222  85.50% | 22s
  ep 08 P2 fine-tune       | tr 0.0542  94.44% | val 0.0783  91.54% | 22s
  ep 09 P2 fine-tune       | tr 0.0367  96.67% | val 0.0799  92.75% | 22s
  ep 10 P2 fine-tune       | tr 0.0298  97.88% | val 0.0680  93.66% | 22s
  ep 11 P2 fine-tune       | tr 0.0325  97.49% | val 0.1009  90.33% | 22s
  ep 12 P2 fine-tune       | tr 0.0272  98.18% | val 0.0704  93.96% | 22s
  ep 13 P2 fine-tune       | tr 0.0284  97.88% | val 0.0503  95.77% | 22s
  ep 14 P2 fine-tune       | tr 0.0257

In [ ]:
# ── 13. Metrics and statistics (all computed from the saved logits) ─────────────────────────────────
MODES = {'single': 'Single view, $T=1$', 'singleT': 'Single view, $T=T^{*}$',
         'tta': 'TTA $K{=}14$, $T=1$', 'ttaT': 'TTA $K{=}14$, $T=T^{*}$ (full)', 'k6T': 'TTA $K{=}6$, $T=T^{*}$'}
TEST_NAMES = {'nick': 'Nickparvar test', 'brisc': 'BRISC test', 'pmram': 'PMRAM'}

def softmax_np(z, T=1.0):
    z = np.asarray(z, np.float64) / T
    z = z - z.max(-1, keepdims=True); e = np.exp(z)
    return e / e.sum(-1, keepdims=True)

def probs_for(L, mode, T):
    if mode == 'single': return softmax_np(L[:, 0])
    if mode == 'singleT': return softmax_np(L[:, 0], T)
    if mode == 'tta': return softmax_np(L).mean(1)
    if mode == 'ttaT': return softmax_np(L, T).mean(1)
    if mode == 'k6T': return softmax_np(L[:, K6_VIEWS], T).mean(1)
    if mode.startswith('view'): return softmax_np(L[:, int(mode[4:])], T)
    raise ValueError(mode)

def wilson(k, n, z=1.959963984540054):
    if n == 0: return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (max(0.0, c - h), min(1.0, c + h))

def ece_score(p, y, bins=None):
    bins = bins or CFG['ece_bins']
    conf, pred = p.max(1), p.argmax(1); acc = (pred == y).astype(float)
    e, edges = 0.0, np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def auc_fast(ybin, score):
    """ROC AUC via the Mann–Whitney rank statistic (ties averaged)."""
    npos = int(ybin.sum()); nneg = len(ybin) - npos
    if npos == 0 or nneg == 0: return np.nan
    r = stats.rankdata(score)
    return float((r[ybin.astype(bool)].sum() - npos * (npos + 1) / 2) / (npos * nneg))

def cm_prf(cm):
    tp = np.diag(cm).astype(float)
    P = np.divide(tp, cm.sum(0), out=np.zeros(4), where=cm.sum(0) > 0)
    R = np.divide(tp, cm.sum(1), out=np.zeros(4), where=cm.sum(1) > 0)
    F = np.divide(2 * P * R, P + R, out=np.zeros(4), where=(P + R) > 0)
    return P, R, F

def metrics(y, p):
    y = np.asarray(y); pred = p.argmax(1); n = len(y)
    cm = np.bincount(y * 4 + pred, minlength=16).reshape(4, 4)
    P, R, F = cm_prf(cm); present = [c for c in range(4) if cm[c].sum() > 0]
    aucs = [auc_fast((y == c).astype(int), p[:, c]) for c in range(4)]
    k = int((pred == y).sum()); lo, hi = wilson(k, n)
    onehot = np.eye(4)[y]
    return dict(n=n, correct=k, errors=n - k, acc=k / n, acc_lo=lo, acc_hi=hi,
                f1=float(np.mean(F[present])) if present else np.nan,
                prec=float(np.mean(P[present])) if present else np.nan,
                rec=float(np.mean(R[present])) if present else np.nan,
                auc=float(np.nanmean([aucs[c] for c in present])) if len(present) > 1 else np.nan,
                ece=ece_score(p, y), brier=float(np.mean(np.sum((p - onehot) ** 2, 1))),
                nll=float(-np.mean(np.log(np.clip(p[np.arange(n), y], 1e-12, 1)))),
                P=P, R=R, F=F, AUC=np.array(aucs), support=cm.sum(1), cm=cm,
                R_ci=[wilson(int(cm[c, c]), int(cm[c].sum())) for c in range(4)])

def bootstrap_ci(y, p, B=None, seed=None):
    """Percentile bootstrap (resampling test images) for macro-F1, macro-AUC and per-class P/R/F1/AUC."""
    B = B or CFG['bootstrap_b']; rng = np.random.default_rng(CFG['stat_seed'] if seed is None else seed)
    y = np.asarray(y); pred = p.argmax(1); n = len(y)
    out = defaultdict(list)
    for _ in range(B):
        idx = rng.integers(0, n, n); yb, pb = y[idx], pred[idx]
        cm = np.bincount(yb * 4 + pb, minlength=16).reshape(4, 4)
        P, R, F = cm_prf(cm); present = [c for c in range(4) if cm[c].sum() > 0]
        out['f1'].append(np.mean(F[present]))
        aucs = [auc_fast((yb == c).astype(int), p[idx, c]) for c in range(4)]
        out['auc'].append(np.nanmean([aucs[c] for c in present]) if len(present) > 1 else np.nan)
        for c in range(4):
            out[f'P{c}'].append(P[c]); out[f'R{c}'].append(R[c]); out[f'F{c}'].append(F[c]); out[f'AUC{c}'].append(aucs[c])
    return {k: tuple(np.nanpercentile(np.array(v, float), [2.5, 97.5])) for k, v in out.items()}

def mcnemar_exact(c1, c2):
    """Exact two-sided McNemar test on paired correctness vectors (same test images)."""
    c1, c2 = np.asarray(c1, bool), np.asarray(c2, bool)
    b, c = int((c1 & ~c2).sum()), int((~c1 & c2).sum())
    return (1.0 if b + c == 0 else float(binomtest(min(b, c), b + c, 0.5).pvalue)), b, c

# ── LaTeX helpers ──────────────────────────────────────────────────────────────────────────────────
MAC = {}
def tex_escape(s):
    return (str(s).replace('\\', r'\textbackslash{}').replace('&', r'\&').replace('%', r'\%')
            .replace('_', r'\_').replace('#', r'\#'))
def setm(key, val, fmt=None, raw=False):
    """Store a manuscript macro. Strings are LaTeX-escaped unless raw=True (already valid LaTeX)."""
    if val is None or (isinstance(val, (float, np.floating)) and not np.isfinite(val)): MAC[key] = 'n/a'; return
    if fmt: MAC[key] = fmt.format(val)
    elif isinstance(val, str): MAC[key] = val if raw else tex_escape(val)
    else: MAC[key] = str(val)
def pc(x, d=2): return '--' if x is None or not np.isfinite(x) else f'{100 * x:.{d}f}'
def ci(lo, hi, d=2): return '--' if not (np.isfinite(lo) and np.isfinite(hi)) else f'[{100 * lo:.{d}f}, {100 * hi:.{d}f}]'
def fnum(x, d=4): return '--' if x is None or not np.isfinite(x) else f'{x:.{d}f}'
def pfmt(p): return '--' if p is None or not np.isfinite(p) else ('$<$0.001' if p < 0.001 else f'{p:.3f}')
def ncols(colspec):
    return sum(ch in 'lcrp' for ch in re.sub(r'\{[^{}]*\}', '', colspec))
def write_tab(name, colspec, header_rows, body_rows, note=None):
    lines = [r'\begin{tabular}{' + colspec + '}', r'\toprule'] + header_rows + [r'\midrule']
    for r in body_rows:
        rule_only = re.fullmatch(r'\s*(\\(mid|bottom|top)rule|\\cmidrule(\([lr]+\))?\{[\d-]+\}|\\addlinespace)(\s+\\cmidrule(\([lr]+\))?\{[\d-]+\})*\s*', r)
        lines.append(r if rule_only or r.rstrip().endswith('\\\\') else r + r' \\')
    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    (DIRS['generated'] / f'{name}.tex').write_text('\n'.join(lines) + '\n')
    # table notes go to a separate file: the manuscript places them with \footnotetext below the tabular
    (DIRS['generated'] / f'{name}_note.tex').write_text((note or '') + '\n')

In [ ]:
# ── 14. Load every finished run ─────────────────────────────────────────────────────────────────────
RUNS = {}
for rc in RUN_PLAN:
    rdir = DIRS['runs'] / rc['name']
    if not (rdir / 'DONE').exists(): continue
    L = {}
    for f in rdir.glob('logits_*_*.npz'):
        _, m, t = f.stem.split('_', 2); z = np.load(f)
        L[(m, t)] = (z['logits'], z['labels'].astype(int), z['uids'])
    RUNS[rc['name']] = dict(cfg=rc, summary=json.load(open(rdir / 'summary.json')),
                            history=json.load(open(rdir / 'history.json')), logits=L,
                            val=dict(np.load(rdir / 'val_logits.npz')))
assert 'main' in RUNS, 'the main run has not finished yet'
print('runs available:', list(RUNS))

def sel_logits(run, t, model=None):
    R_ = RUNS[run]; m = model or R_['summary']['selected']
    return R_['logits'].get((m, t))

def run_T(run, model=None):
    S = RUNS[run]['summary']; m = model or S['selected']
    return S['T_swa'] if m == 'swa' else S['T_best']

TESTS = [t for t in ['nick', 'brisc', 'pmram'] if sel_logits('main', t) is not None]

runs available: ['main', 'abl_no_lgg', 'abl_no_brisc', 'abl_nick_only', 'abl_no_boost', 'abl_uniform', 'abl_percentile', 'main_patient_disjoint']


In [ ]:
# ── 15. Dataset / protocol numbers, overlap table, partition table ───────────────────────────────────
N = MAN.source.value_counts().to_dict()
setm('n_nick_train_raw', N.get('nick_train', 0), '{:,}'); setm('n_nick_test', N.get('nick_test', 0), '{:,}')
setm('n_brisc_train_raw', N.get('brisc_train', 0), '{:,}'); setm('n_brisc_test', N.get('brisc_test', 0), '{:,}')
setm('n_pmram', N.get('pmram', 0), '{:,}'); setm('n_lgg_slices', N.get('lgg', 0), '{:,}')
setm('n_lgg_patients', MAN.loc[MAN.source == 'lgg', 'pid'].nunique(), '{:,}')
setm('n_lgg_tumour_slices', int((PART.role == 'train_lgg_slice').sum()), '{:,}')
setm('n_lgg_notumour_slices', int((PART.role == 'probe_lgg_notumour').sum()), '{:,}')
setm('n_lgg_crops', len(LGG_CROPS['ws']), '{:,}')
RC = PART.role.value_counts().to_dict()
setm('n_val', RC.get('val', 0), '{:,}'); setm('n_train_t1', RC.get('train', 0), '{:,}')
setm('n_train_total', RC.get('train', 0) + len(LGG_CROPS['ws']), '{:,}')
for s in ['nick_train', 'brisc_train', 'lgg']:
    setm(f'n_excl_dup_test_{s}', int(((PART.role == 'excluded_dup_of_test') & (PART.source == s)).sum()), '{:,}')
setm('n_excl_dup_test', RC.get('excluded_dup_of_test', 0), '{:,}')
setm('n_excl_dup_within', RC.get('excluded_dup_within_train', 0), '{:,}')
setm('n_excl_label_conflict', RC.get('excluded_label_conflict', 0), '{:,}')
setm('n_label_conflict_pairs', len(LABEL_CONFLICTS), '{:,}')
setm('dup_ncc', CFG['dup_ncc'], '{:.2f}'); setm('dup_bp', CFG['dup_bp'], '{:.2f}')
setm('dup_exact_bp', CFG['dup_exact_bp'], '{:.2f}'); setm('pair_raw_min', CFG['pair_raw_min'], '{:.2f}')
setm('n_pairs_all', len(SCR) * (len(SCR) - 1) // 2, '{:,}'); setm('n_pairs_related', int(PAIRS.dup.sum()), '{:,}')
setm('n_pairs_exact', int(is_dup(PAIRS.ncc.values, PAIRS.bp.values, CFG['dup_exact_bp']).sum()), '{:,}')
setm('n_related_label_mismatch', len(REL_MISMATCH), '{:,}')
setm('n_related_t1_pairs', int(((PAIRS.dup) & (PAIRS.src_i != 'lgg') & (PAIRS.src_j != 'lgg')).sum()), '{:,}')
setm('dup_bp_lo', min(CFG['dup_bp_grid']), '{:.2f}'); setm('dup_bp_hi', max(CFG['dup_bp_grid']), '{:.2f}')
setm('val_frac_pct', 100 * CFG['val_frac'], '{:.0f}')
pre = json.load(open(DIRS['results'] / 'preprocessing.json'))
setm('ws_fallback_pct', 100 * pre['fallback_rate'], '{:.1f}')
setm('ws_fallback_t1_pct', 100 * pre['fallback_rate_T1'], '{:.1f}'); setm('ws_fallback_t2_pct', 100 * pre['fallback_rate_T2'], '{:.1f}')
for c in ['T1', 'T2']:
    setm(f'ws_win_{c.lower()}_lo', pre['window'][c][0], '{:.1f}'); setm(f'ws_win_{c.lower()}_hi', pre['window'][c][1], '{:.1f}')

SRC_LABEL = {'nick_train': 'Nickparvar-train', 'nick_test': 'Nickparvar-test', 'brisc_train': 'BRISC-train',
             'brisc_test': 'BRISC-test', 'lgg': 'LGG (FLAIR)', 'pmram': 'PMRAM'}
o = OVL[CFG['dup_bp']]
for A in EVAL_SOURCES:
    nA = N_SRC.get(A, 0)
    for B in SCREEN + ['any_train']:
        setm(f'ov_{A}__{B}_n', o[(A, B)], '{:,}'); setm(f'ov_{A}__{B}_pct', 100 * o[(A, B)] / max(nA, 1), '{:.1f}')
    for thr in CFG['dup_bp_grid']:
        setm(f'ov_{A}__any_train_n_{int(round(thr * 100)):03d}', OVL[thr][(A, 'any_train')], '{:,}')
for s in ['nick_train', 'nick_test', 'brisc_train', 'brisc_test']:
    setm(f'dup_within_{s}_n', o[(s, s)], '{:,}')
rows = []
for B in ['nick_train', 'brisc_train', 'lgg', 'any_train'] + EVAL_SOURCES:
    lab = r'\textit{Any training source}' if B == 'any_train' else SRC_LABEL[B]
    cells = [lab]
    for A in EVAL_SOURCES:
        nA = N_SRC.get(A, 0)
        cells.append('--' if A == B else f'{o[(A, B)]:,} ({100 * o[(A, B)] / max(nA, 1):.1f}\\%)')
    rows.append(' & '.join(cells))
    if B == 'any_train': rows.append(r'\midrule')
write_tab('tab_overlap', 'l ' + 'r ' * len(EVAL_SOURCES),
          [r'\textbf{Near-duplicate found in} & ' + ' & '.join(r'\textbf{' + SRC_LABEL[A] + '}' for A in EVAL_SOURCES) + r' \\',
           ' & '.join([''] + [f'$n={N_SRC.get(A, 0):,}$' for A in EVAL_SOURCES]) + r' \\'],
          rows, note=(f'Number (\\%) of test images with at least one copy or closely related image (e.g. a re-cropped copy '
                      f'or an adjacent slice) in the source named in the row. All image pairs were compared (brain-cropped '
                      f'64$\\times$64 thumbnails, mirrored orientation included); a pair is linked when the normalised '
                      f'cross-correlation is $\\ge$ {CFG["dup_ncc"]:.2f} and the band-pass (difference-of-Gaussians) '
                      f'correlation is $\\ge$ {CFG["dup_bp"]:.2f}. Every training image linked to a test image was removed '
                      'before training; the lower rows show overlap between test sets.'))
grid_rows = []
for A in EVAL_SOURCES:
    grid_rows.append(' & '.join([SRC_LABEL[A], f'{N_SRC.get(A, 0):,}'] + [f'{OVL[t][(A, "any_train")]:,}' for t in CFG['dup_bp_grid']]))
write_tab('tab_overlap_grid_supp', 'l r ' + 'r ' * len(CFG['dup_bp_grid']),
          [r'\textbf{Test set} & $n$ & ' + ' & '.join(f'band-pass $\\ge$ {t:.2f}' for t in CFG['dup_bp_grid']) + r' \\'], grid_rows,
          note=f'Number of test images linked to any training source as the band-pass threshold varies '
               f'(raw NCC $\\ge$ {CFG["dup_ncc"]:.2f} throughout). The primary analysis uses {CFG["dup_bp"]:.2f}; '
               f'$\\ge$ {CFG["dup_exact_bp"]:.2f} corresponds to exact copies.')

# partition table
role_order = [('train', 'Training (T1-CE)'), ('train_lgg_slice', r'Training (LGG FLAIR slices $\rightarrow$ crops)'),
              ('val', 'Validation (selection + calibration)'), ('test_nick', 'Test: Nickparvar'),
              ('test_brisc', 'Test: BRISC'), ('test_pmram', 'Test: PMRAM'),
              ('probe_lgg_notumour', 'Probe: LGG tumour-free FLAIR'),
              ('excluded_dup_of_test', 'Removed: linked to a test image'),
              ('excluded_dup_within_train', 'Removed: exact copy within training pool'),
              ('excluded_label_conflict', 'Removed: exact copies with conflicting labels')]
prow = []
for role, lab in role_order:
    d = PART[PART.role == role]
    if not len(d): continue
    cc = d.label.value_counts().reindex(range(4), fill_value=0).tolist()
    src = ', '.join(f'{SRC_LABEL[s]} {n:,}' for s, n in d.source.value_counts().items())
    npid = d.pid.fillna('').ne('').sum()
    prow.append(' & '.join([lab] + [f'{x:,}' for x in cc] + [f'{len(d):,}', f'{npid:,}', src]))
write_tab('tab_partition', 'p{3.0cm} r r r r r r p{2.9cm}',
          [r'\textbf{Role} & \textbf{Gli.} & \textbf{Men.} & \textbf{Pit.} & \textbf{NoT.} & \textbf{Total} & '
           r'\textbf{PID} & \textbf{Sources} \\'], prow,
          note='PID = images with a known patient identifier (TCGA patient for LGG; figshare patient ID for '
               'Nickparvar/BRISC images traced to the Cheng et al. release). Numbers for LGG are slices; each '
               f'tumour-bearing slice yields {CFG["lgg_crops_per"]} crops.')
setm('lgg_crops_per', CFG['lgg_crops_per'])

# figshare tracing / patient numbers
fsd = PART[PART.pid.fillna('').str.startswith('FS:')]
setm('fs_patients', fsd.pid.nunique(), '{:,}'); setm('fs_traced_n', len(fsd), '{:,}')
for s in ['nick_train', 'nick_test', 'brisc_train', 'brisc_test']:
    setm(f'fs_traced_{s}', int((fsd.source == s).sum()), '{:,}')
setm('fs_label_agree_pct', 100 * float((SCR.fs_label.isna() | (SCR.fs_label == SCR.label)).mean()), '{:.1f}')
trpids = set(PART.loc[PART.role.isin(['train', 'val']), 'pid'].dropna()) - {''}
setm('test_pids_n', len(TEST_PIDS), '{:,}'); setm('test_pids_in_train_n', len(TEST_PIDS & trpids), '{:,}')

In [ ]:
# ── 16. Main model: test-set results, per-class tables, inference decomposition, SWA/TTA/T ablations ───
S0 = RUNS['main']['summary']; T0 = run_T('main')
setm('main_selected', 'SWA model' if S0['selected'] == 'swa' else 'best single checkpoint')
setm('main_best_epoch', S0['best_epoch']); setm('main_valacc_best', S0['val_acc_best'], '{:.2f}')
setm('main_valacc_swa', S0['val_acc_swa'], '{:.2f}'); setm('main_T', T0, '{:.4f}')
setm('main_train_min', S0['train_minutes'], '{:.0f}'); setm('main_epoch_sec', S0['epoch_seconds'], '{:.0f}')
setm('main_ntrain', S0['n_train'], '{:,}'); setm('main_nval', S0['n_val'], '{:,}')
setm('gpu_name', S0['gpu']); setm('n_epochs', CFG['warmup_epochs'] + CFG['finetune_epochs'])
setm('total_gpu_hours', sum(R_['summary']['total_minutes'] for R_ in RUNS.values()) / 60, '{:.1f}')

RES = {}
for t in TESTS:
    L, y, _ = sel_logits('main', t)
    for mode in MODES:
        m = metrics(y, probs_for(L, mode, T0)); RES[(t, mode)] = m
        for k in ['acc', 'acc_lo', 'acc_hi', 'f1', 'prec', 'rec', 'ece']:
            setm(f'main_{t}_{mode}_{k}', 100 * m[k], '{:.2f}')
        setm(f'main_{t}_{mode}_auc', m['auc'], '{:.4f}'); setm(f'main_{t}_{mode}_brier', m['brier'], '{:.4f}')
        setm(f'main_{t}_{mode}_nll', m['nll'], '{:.4f}'); setm(f'main_{t}_{mode}_errors', m['errors'], '{:,}')
    setm(f'main_{t}_n', len(y), '{:,}')
    bs = bootstrap_ci(y, probs_for(L, 'ttaT', T0)); RES[(t, 'boot')] = bs
    setm(f'main_{t}_ttaT_f1_lo', 100 * bs['f1'][0], '{:.2f}'); setm(f'main_{t}_ttaT_f1_hi', 100 * bs['f1'][1], '{:.2f}')
    setm(f'main_{t}_ttaT_auc_lo', bs['auc'][0], '{:.4f}'); setm(f'main_{t}_ttaT_auc_hi', bs['auc'][1], '{:.4f}')
    m = RES[(t, 'ttaT')]
    for c, nm in enumerate(['gl', 'me', 'pi', 'nt']):
        setm(f'main_{t}_ttaT_{nm}_recall', 100 * m['R'][c], '{:.2f}'); setm(f'main_{t}_ttaT_{nm}_f1', 100 * m['F'][c], '{:.2f}')
    # confusion: most frequent error type
    cmo = m['cm'].copy(); np.fill_diagonal(cmo, 0)
    if cmo.sum():
        a, b = np.unravel_index(np.argmax(cmo), cmo.shape)
        setm(f'main_{t}_top_error', f'{CLASSES[a]} $\\rightarrow$ {CLASSES[b]} ({cmo[a, b]} of {cmo.sum()} errors)', raw=True)
    else:
        setm(f'main_{t}_top_error', 'none')
    for a_, b_, key in [('single', 'ttaT', 'single_vs_ttaT'), ('tta', 'ttaT', 'tta_vs_ttaT'), ('single', 'tta', 'single_vs_tta')]:
        pa = RES[(t, a_)]; pb = RES[(t, b_)]
        ca = probs_for(L, a_, T0).argmax(1) == y; cb = probs_for(L, b_, T0).argmax(1) == y
        p, b1, c1 = mcnemar_exact(ca, cb); setm(f'mcn_{t}_{key}_p', pfmt(p), raw=True)
    # SWA vs best checkpoint (both through the full pipeline, each with its own validation-fitted T)
    other = 'best' if S0['selected'] == 'swa' else 'swa'
    Lo = sel_logits('main', t, other)
    if Lo is not None:
        mo = metrics(Lo[1], probs_for(Lo[0], 'ttaT', run_T('main', other))); RES[(t, 'other')] = mo
        p, _, _ = mcnemar_exact(probs_for(L, 'ttaT', T0).argmax(1) == y, probs_for(Lo[0], 'ttaT', run_T('main', other)).argmax(1) == Lo[1])
        setm(f'main_{t}_other_acc', 100 * mo['acc'], '{:.2f}'); setm(f'mcn_{t}_swa_vs_best_p', pfmt(p), raw=True)
setm('main_other_name', 'best single checkpoint' if S0['selected'] == 'swa' else 'SWA model')

# PMRAM images with NO copy or related image in any public source (independence of the external set, R2.4)
PUB = ['nick_train', 'nick_test', 'brisc_train', 'brisc_test', 'lgg']
PM_KEEP = None
if 'pmram' in TESTS:
    _cf = PAIRS[PAIRS.dup]
    _a = np.concatenate([_cf.i.values, _cf.j.values]); _b = np.concatenate([_cf.j.values, _cf.i.values])
    PM_LINKED = set(SCR.uid.values[_a[(SCR.source.values[_a] == 'pmram') & np.isin(SCR.source.values[_b], PUB)]])
    L, y, u = sel_logits('main', 'pmram'); PM_KEEP = ~np.isin(u, list(PM_LINKED))
    setm('n_pmram_linked', int((~PM_KEEP).sum()), '{:,}'); setm('pct_pmram_linked', 100 * (~PM_KEEP).mean(), '{:.1f}')
    for c, nm in enumerate(['gl', 'me', 'pi', 'nt']):
        setm(f'pmram_linked_{nm}', int(((~PM_KEEP) & (y == c)).sum()), '{:,}')
        setm(f'pmram_linked_{nm}_pct', 100 * ((~PM_KEEP) & (y == c)).sum() / max((y == c).sum(), 1), '{:.1f}')
    pd.DataFrame(dict(uid=u, label=y, public_match=~PM_KEEP)).to_csv(DIRS['results'] / 'pmram_public_matches.csv', index=False)
    print(f'PMRAM images with a copy/related image in a public source: {int((~PM_KEEP).sum())} of {len(u)}')
    if PM_KEEP.sum() >= 20:
        Lu, yu = L[PM_KEEP], y[PM_KEEP]
        for mode in MODES:
            m = metrics(yu, probs_for(Lu, mode, T0)); RES[('pmramu', mode)] = m
            for k in ['acc', 'acc_lo', 'acc_hi', 'f1', 'prec', 'rec', 'ece']:
                setm(f'main_pmramu_{mode}_{k}', 100 * m[k], '{:.2f}')
            setm(f'main_pmramu_{mode}_auc', m['auc'], '{:.4f}'); setm(f'main_pmramu_{mode}_brier', m['brier'], '{:.4f}')
            setm(f'main_pmramu_{mode}_errors', m['errors'], '{:,}')
        setm('main_pmramu_n', int(PM_KEEP.sum()), '{:,}')
        RES[('pmramu', 'boot')] = bootstrap_ci(yu, probs_for(Lu, 'ttaT', T0))
        m = RES[('pmramu', 'ttaT')]
        setm('main_pmramu_ttaT_f1_lo', 100 * RES[('pmramu', 'boot')]['f1'][0], '{:.2f}')
        setm('main_pmramu_ttaT_f1_hi', 100 * RES[('pmramu', 'boot')]['f1'][1], '{:.2f}')
        for c, nm in enumerate(['gl', 'me', 'pi', 'nt']):
            n_c = int(m['support'][c]); setm(f'main_pmramu_n_{nm}', n_c, '{:,}')
            setm(f'main_pmramu_ttaT_{nm}_recall', 100 * m['R'][c] if n_c else None, '{:.2f}')

# main results table (rows = metrics, columns = test sets)
TAB_T = TESTS + (['pmramu'] if ('pmramu', 'ttaT') in RES else [])
HDR = {'nick': ('Nickparvar', 'test'), 'brisc': ('BRISC', 'test'), 'pmram': ('PMRAM', 'all'),
       'pmramu': ('PMRAM', 'unmatched')}
def _cells(fn): return ' & '.join(fn(t) for t in TAB_T)
rows = [r'$n$ & ' + _cells(lambda t: f"{RES[(t, 'ttaT')]['n']:,}"),
        r'Accuracy (\%) & ' + _cells(lambda t: pc(RES[(t, 'ttaT')]['acc'])),
        r'\quad 95\% CI (Wilson) & ' + _cells(lambda t: ci(RES[(t, 'ttaT')]['acc_lo'], RES[(t, 'ttaT')]['acc_hi'])),
        r'Macro-F1 (\%) & ' + _cells(lambda t: pc(RES[(t, 'ttaT')]['f1'])),
        r'\quad 95\% CI (bootstrap) & ' + _cells(lambda t: ci(*RES[(t, 'boot')]['f1'])),
        r'Macro-AUC & ' + _cells(lambda t: fnum(RES[(t, 'ttaT')]['auc'])),
        r'\quad 95\% CI (bootstrap) & ' + _cells(lambda t: f"[{fnum(RES[(t, 'boot')]['auc'][0])}, {fnum(RES[(t, 'boot')]['auc'][1])}]"),
        r'ECE (\%) & ' + _cells(lambda t: pc(RES[(t, 'ttaT')]['ece'])),
        r'Brier score & ' + _cells(lambda t: fnum(RES[(t, 'ttaT')]['brier'])),
        r'Misclassified images & ' + _cells(lambda t: f"{RES[(t, 'ttaT')]['errors']:,}")]
write_tab('tab_main', 'l ' + 'r ' * len(TAB_T),
          [r'\textbf{Metric} & ' + ' & '.join(r'\textbf{' + HDR[t][0] + '}' for t in TAB_T) + r' \\',
           ' & ' + ' & '.join(r'\textbf{' + HDR[t][1] + '}' for t in TAB_T) + r' \\'], rows,
          note='Full pipeline: selected model, TTA with $K{=}14$ views, temperature $T^{*}$ fitted on the validation split. '
               f'Bootstrap: $B={CFG["bootstrap_b"]:,}$ resamples of the test images.'
               + (' PMRAM unmatched: the PMRAM images without a copy or closely related image in Nickparvar, BRISC '
                  'or LGG.' if 'pmramu' in TAB_T else ''))

# per-class table (all test sets stacked)
rows = []
for t in TESTS:
    m, bs = RES[(t, 'ttaT')], RES[(t, 'boot')]
    rows.append(r'\multicolumn{6}{l}{\textit{' + TEST_NAMES[t] + f' ($n={m["n"]:,}$)' + r'}} \\')
    for c in range(4):
        if m['support'][c] == 0: continue
        rows.append(' & '.join([r'\quad ' + CLASSES[c], pc(m['P'][c]),
                                f"{pc(m['R'][c])} {ci(*m['R_ci'][c], d=1)}", f"{pc(m['F'][c])} {ci(*bs[f'F{c}'], d=1)}",
                                fnum(m['AUC'][c]), f"{int(m['support'][c]):,}"]))
write_tab('tab_perclass', 'l c c c c r',
          [r'\textbf{Class} & \textbf{Precision (\%)} & \textbf{Recall (\%) [95\% CI]} & \textbf{F1 (\%) [95\% CI]} & \textbf{AUC} & \textbf{$n$} \\'],
          rows, note='Full pipeline (TTA $K{=}14$ + $T^{*}$). Recall CIs: Wilson score; F1 CIs: percentile bootstrap. '
                     'Bootstrap CIs for precision and per-class AUC are provided in Supplementary Table S3.')
rows = []
for t in TESTS:
    m, bs = RES[(t, 'ttaT')], RES[(t, 'boot')]
    rows.append(r'\multicolumn{3}{l}{\textit{' + TEST_NAMES[t] + r'}} \\')
    for c in range(4):
        if m['support'][c] == 0: continue
        rows.append(' & '.join([r'\quad ' + CLASSES[c], f"{pc(m['P'][c])} {ci(*bs[f'P{c}'], d=1)}",
                                f"{fnum(m['AUC'][c])} [{fnum(bs[f'AUC{c}'][0])}, {fnum(bs[f'AUC{c}'][1])}]"]))
write_tab('tab_perclass_ci_supp', 'l c c',
          [r'\textbf{Class} & \textbf{Precision (\%) [95\% CI]} & \textbf{AUC [95\% CI]} \\'], rows,
          note='Percentile bootstrap over test images.')

# inference decomposition
rows = []
for mode in ['single', 'singleT', 'tta', 'ttaT', 'k6T']:
    cells = [MODES[mode]]
    for t in [x for x in TESTS if x in ('nick', 'brisc')]:
        m = RES[(t, mode)]
        cells += [f"{pc(m['acc'])}", pc(m['ece']), fnum(m['brier'])]
    rows.append(' & '.join(cells))
nt = len([x for x in TESTS if x in ('nick', 'brisc')])
write_tab('tab_inference', 'l ' + 'r r r ' * nt,
          [' & '.join([''] + [r'\multicolumn{3}{c}{\textbf{' + TEST_NAMES[t] + '}}' for t in TESTS if t in ('nick', 'brisc')]) + r' \\',
           ' '.join(r'\cmidrule(lr){' + f'{2 + 3 * i}-{4 + 3 * i}' + '}' for i in range(nt)),
           ' & '.join([r'\textbf{Inference}'] + ['Acc (\\%)', 'ECE (\\%)', 'Brier'] * nt) + r' \\'],
          rows, note=f'Same trained model throughout; $T^{{*}}={T0:.4f}$ was fitted on the validation split. '
                     'Temperature scaling cannot change a single-view prediction (argmax is invariant to a positive '
                     'scalar) but can change a TTA prediction because probabilities are averaged after scaling.')

# per-view accuracy (supplementary)
rows = []
for k in range(CFG['tta_k']):
    cells = [f'{k + 1}', TTA_NAMES[k]]
    for t in [x for x in TESTS if x in ('nick', 'brisc')]:
        L, y, _ = sel_logits('main', t); cells.append(pc((probs_for(L, f'view{k}', T0).argmax(1) == y).mean()))
    rows.append(' & '.join(cells))
write_tab('tab_perview_supp', 'r l ' + 'r ' * nt,
          [r'\textbf{\#} & \textbf{View} & ' + ' & '.join(r'\textbf{' + TEST_NAMES[t] + ' (\\%)}' for t in TESTS if t in ('nick', 'brisc')) + r' \\'], rows)
for t in [x for x in TESTS if x in ('nick', 'brisc')]:
    L, y, _ = sel_logits('main', t)
    accs = np.array([(probs_for(L, f'view{k}', T0).argmax(1) == y).mean() for k in range(CFG['tta_k'])])
    setm(f'perview_{t}_min', 100 * accs.min(), '{:.2f}'); setm(f'perview_{t}_max', 100 * accs.max(), '{:.2f}')
    setm(f'perview_{t}_rot_mean', 100 * accs[[4, 5, 6, 12]].mean(), '{:.2f}')
    setm(f'perview_{t}_worst', TTA_NAMES[int(accs.argmin())])

PMRAM images with a copy/related image in a public source: 981 of 1505


In [ ]:
# ── 17. Controlled ablations, patient-level analysis, shortcut probe, seeds ────────────────────────────
from scipy.stats import fisher_exact

def aligned(run, t, model=None):
    """Logits of `run` on test set t, re-ordered to the main run's image order."""
    x = sel_logits(run, t, model)
    if x is None: return None
    L, y, u = x; Lm, ym, um = sel_logits('main', t)
    if len(u) == len(um) and np.array_equal(u, um): return L, y
    pos = {k: i for i, k in enumerate(u)}; idx = np.array([pos[k] for k in um])
    return L[idx], y[idx]

def pred_ttaT(run, t, model=None):
    a = aligned(run, t, model)
    return None if a is None else (probs_for(a[0], 'ttaT', run_T(run, model)), a[1])

main_pred = {t: pred_ttaT('main', t) for t in TESTS}
ABL = []   # (label, {t: (probs, y)}, key)
ABL.append(('Full model (reference)', main_pred, 'full'))
ABL_KEY = {'abl_no_lgg': 'no_lgg', 'abl_no_brisc': 'no_brisc', 'abl_nick_only': 'nick_only', 'abl_no_boost': 'no_boost',
           'abl_uniform': 'uniform', 'abl_percentile': 'percentile', 'main_patient_disjoint': 'patient_disjoint'}
for name in ['abl_no_lgg', 'abl_no_brisc', 'abl_nick_only', 'abl_no_boost', 'abl_uniform', 'abl_percentile',
             'main_patient_disjoint']:
    if name in RUNS:
        ABL.append((RUNS[name]['cfg']['label'], {t: pred_ttaT(name, t) for t in TESTS}, ABL_KEY[name]))
other = 'best' if S0['selected'] == 'swa' else 'swa'
if sel_logits('main', 'nick', other) is not None:
    ABL.append(('SWA $\\rightarrow$ best single checkpoint' if other == 'best' else 'Best checkpoint $\\rightarrow$ SWA',
                {t: pred_ttaT('main', t, other) for t in TESTS}, 'swa_other'))
for mode, lab, k_ in [('singleT', 'No TTA (single view, $T^{*}$)', 'no_tta'), ('tta', 'No temperature scaling (TTA, $T=1$)', 'no_temp'),
                      ('k6T', 'TTA with $K{=}6$ views', 'k6')]:
    ABL.append((lab, {t: (probs_for(sel_logits('main', t)[0], mode, T0), sel_logits('main', t)[1]) for t in TESTS}, k_))

def holm(pvals):
    """Holm–Bonferroni adjusted p-values (family = the configurations compared on one test set)."""
    p_ = np.asarray(pvals, float); o = np.argsort(p_); m = len(p_); adj = np.empty(m); run = 0.0
    for r, i in enumerate(o):
        run = max(run, min(1.0, (m - r) * p_[i])); adj[i] = run
    return adj

def stars(p): return '' if not np.isfinite(p) else ('***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else '')

# raw McNemar p per (configuration, test set), then Holm adjustment within each test set
RAWP, ACC_T, DEL_T = {}, {}, {}
for lab, P_, key in ABL:
    for t in TESTS:
        if P_.get(t) is None: continue
        p, y = P_[t]; pm, ym = main_pred[t]
        ACC_T[(key, t)] = (p.argmax(1) == y).mean()
        DEL_T[(key, t)] = 100 * (ACC_T[(key, t)] - (pm.argmax(1) == ym).mean())
        if key != 'full': RAWP[(key, t)] = mcnemar_exact(p.argmax(1) == y, pm.argmax(1) == ym)[0]
if PM_KEEP is not None and PM_KEEP.sum() >= 20:      # accuracy on PMRAM images without a public match
    for lab, P_, key in ABL:
        if P_.get('pmram') is None: continue
        p, y = P_['pmram']; setm(f'abl_{key}_pmramu_acc', 100 * (p.argmax(1) == y)[PM_KEEP].mean(), '{:.2f}')
ADJP = {}
for t in TESTS:
    ks = [k for (k, tt) in RAWP if tt == t]
    for k, a in zip(ks, holm([RAWP[(k, t)] for k in ks])): ADJP[(k, t)] = a

rows, srows, prows = [], [], []
for lab, P_, key in ABL:
    cells, scells, pcells = [lab], [lab], [lab]
    for t in TESTS:
        if (key, t) not in ACC_T: cells += ['--', '--']; pcells += ['--', '--']; continue
        setm(f'abl_{key}_{t}_acc', 100 * ACC_T[(key, t)], '{:.2f}')
        if key == 'full':
            cells += [pc(ACC_T[(key, t)]), '--']; pcells += ['--', '--']
        else:
            d, pr, pa = DEL_T[(key, t)], RAWP[(key, t)], ADJP[(key, t)]
            setm(f'abl_{key}_{t}_delta', d, '{:+.2f}'); setm(f'abl_{key}_{t}_p', pfmt(pr), raw=True)
            setm(f'abl_{key}_{t}_padj', pfmt(pa), raw=True)
            cells += [pc(ACC_T[(key, t)]), f'{d:+.2f}' + (f'$^{{{stars(pa)}}}$' if stars(pa) else '')]
            pcells += [pfmt(pr), pfmt(pa)]
    if P_.get('nick') is not None:
        m = metrics(P_['nick'][1], P_['nick'][0])
        setm(f'abl_{key}_nick_glf1', 100 * m['F'][0], '{:.2f}'); setm(f'abl_{key}_nick_ece', 100 * m['ece'], '{:.2f}')
        setm(f'abl_{key}_nick_f1', 100 * m['f1'], '{:.2f}'); setm(f'abl_{key}_nick_auc', m['auc'], '{:.4f}')
        scells += [pc(m['F'][c]) for c in range(4)] + [pc(m['f1']), fnum(m['auc'])]
    else:
        scells += ['--'] * 6
    rows.append(' & '.join(cells)); srows.append(' & '.join(scells)); prows.append(' & '.join(pcells))
n_retrained = sum(1 for _, _, k in ABL if k in ABL_KEY.values())
for R_ in (rows, srows, prows): R_.insert(1 + n_retrained, r'\midrule'); R_.insert(1, r'\midrule')
hdr1 = ' & '.join([''] + [r'\multicolumn{2}{c}{\textbf{' + TEST_NAMES[t] + '}}' for t in TESTS]) + r' \\'
cm_ = ' '.join(r'\cmidrule(lr){' + f'{2 + 2 * i}-{3 + 2 * i}' + '}' for i in range(len(TESTS)))
hdr2 = ' & '.join([r'\textbf{Configuration}'] + ['Acc (\\%)', '$\\Delta$ (pp)'] * len(TESTS)) + r' \\'
write_tab('tab_ablation', 'p{3.4cm} ' + 'r r ' * len(TESTS), [hdr1, cm_, hdr2], rows,
          note='Every retrained configuration (middle block) uses the identical schedule (5 warm-up + 25 fine-tuning epochs, '
               'SWA over the last 5), the same validation split for checkpoint selection and temperature fitting, and TTA '
               '$K{=}14$ with its own $T^{*}$; the last block re-uses the trained full model without retraining. $\\Delta$: '
               'accuracy difference to the full model (percentage points). Exact McNemar tests on the same test images, '
               'Holm-adjusted within each test set: $^{*}p<0.05$, $^{**}p<0.01$, $^{***}p<0.001$; raw and adjusted '
               '$p$-values are in Supplementary Table S5.')
write_tab('tab_ablation_supp', 'p{3.4cm} r r r r r r',
          [r'\textbf{Configuration} & \textbf{Gli.} & \textbf{Men.} & \textbf{Pit.} & \textbf{NoT.} & '
           r'\textbf{Macro} & \textbf{Macro-} \\',
           r' & \textbf{F1} & \textbf{F1} & \textbf{F1} & \textbf{F1} & \textbf{F1} & \textbf{AUC} \\'], srows,
          note='Nickparvar test set; F1 in \\%.')
write_tab('tab_ablation_p_supp', 'p{3.4cm} ' + 'r r ' * len(TESTS),
          [' & '.join([''] + [r'\multicolumn{2}{c}{\textbf{' + TEST_NAMES[t] + '}}' for t in TESTS]) + r' \\', cm_,
           ' & '.join([r'\textbf{Configuration}'] + ['$p$', '$p_{\\mathrm{Holm}}$'] * len(TESTS)) + r' \\'], prows,
          note='Exact two-sided McNemar test against the full model on identical test images; Holm--Bonferroni adjustment '
               'over the configurations compared on the same test set.')

# patient-level analysis (figshare-traced test images) ---------------------------------------------
train_pids_main = set(PART.loc[PART.role.isin(['train', 'val']), 'pid'].dropna()) - {''}
uid2pid = dict(zip(PART.uid, PART.pid.fillna('')))
prow = []
for t in [x for x in TESTS if x in ('nick', 'brisc')]:
    _, _, um = sel_logits('main', t)
    pids = np.array([uid2pid.get(u, '') for u in um])
    traced = pids != ''; seen = traced & np.isin(pids, list(train_pids_main)); unseen = traced & ~seen
    subsets = [('all', np.ones(len(um), bool)), ('seen', seen), ('unseen', unseen), ('untraced', ~traced)]
    prow.append(r'\multicolumn{6}{l}{\textit{' + TEST_NAMES[t] + r'}} \\')
    prow.append(' & '.join([r'\quad Images ($n$)'] + [f'{int(m_.sum()):,}' for _, m_ in subsets] + ['']))
    for run, tag, lab in [('main', 'main', 'Full model'), ('main_patient_disjoint', 'pdj', 'Patient-disjoint model')]:
        if run not in RUNS: continue
        p, y = pred_ttaT(run, t); corr = p.argmax(1) == y
        cells = [r'\quad ' + lab + r' (acc., \%)']
        for nm, m_ in subsets:
            k, n = int(corr[m_].sum()), int(m_.sum())
            setm(f'pat_{tag}_{t}_{nm}_n', n, '{:,}'); setm(f'pat_{tag}_{t}_{nm}_acc', 100 * k / n if n else np.nan, '{:.2f}')
            cells.append('--' if n == 0 else f'{100 * k / n:.2f}')
        if seen.sum() and unseen.sum():
            _, pf = fisher_exact([[corr[seen].sum(), (~corr[seen]).sum()], [corr[unseen].sum(), (~corr[unseen]).sum()]])
            setm(f'pat_{tag}_{t}_fisher_p', pfmt(pf), raw=True); cells.append(pfmt(pf))
        else:
            cells.append('--')
        prow.append(' & '.join(cells))
    setm(f'pat_{t}_traced_n', int(traced.sum()), '{:,}'); setm(f'pat_{t}_seen_n', int(seen.sum()), '{:,}')
    setm(f'pat_{t}_unseen_n', int(unseen.sum()), '{:,}')
if 'main_patient_disjoint' in RUNS:
    setm('pdj_excluded_n', RUNS['main_patient_disjoint']['summary']['n_excluded_patient'], '{:,}')
    setm('pdj_ntrain', RUNS['main_patient_disjoint']['summary']['n_train'], '{:,}')
write_tab('tab_patient', 'p{3.6cm} r r r r r',
          [r' & \textbf{All} & \textbf{Seen} & \textbf{Unseen} & \textbf{Untraced} & $p$ \\',
           r' & & \textbf{patient} & \textbf{patient} & & \\'], prow,
          note="``Seen patient'': the figshare patient ID of the test image also occurs in the training or validation data "
               "of the full model; ``Unseen'': traced to a patient absent from them; ``Untraced'': not traceable to the "
               'figshare release (SARTAJ/Br35H-derived images, for which no patient IDs exist). $p$: Fisher exact test, '
               'seen vs unseen. The patient-disjoint model was trained after removing every training/validation image '
               'whose patient appears in any test set.')

# shortcut probe: tumour-free LGG FLAIR slices ------------------------------------------------------
prob_rows = []
for run in ['main', 'abl_no_lgg', 'abl_no_brisc', 'abl_nick_only']:
    if run not in RUNS or sel_logits(run, 'probe') is None: continue
    L, y, _ = sel_logits(run, 'probe'); pr = probs_for(L, 'ttaT', run_T(run)).argmax(1)
    frac = np.bincount(pr, minlength=4) / len(pr)
    key = 'main' if run == 'main' else run
    setm(f'probe_{key}_glioma_pct', 100 * frac[0], '{:.1f}'); setm(f'probe_{key}_notumour_pct', 100 * frac[3], '{:.1f}')
    gl = RES[('nick', 'ttaT')]['R'][0] if run == 'main' else metrics(*pred_ttaT(run, 'nick')[::-1])['R'][0]
    prob_rows.append(' & '.join([RUNS[run]['cfg']['label'], 'yes' if 'lgg' in RUNS[run]['cfg']['sources'] else 'no']
                                + [f'{100 * f:.1f}' for f in frac] + [pc(gl)]))
    setm('probe_n', len(pr), '{:,}')
write_tab('tab_probe', 'p{3.1cm} c r r r r r',
          [r'& & \multicolumn{4}{c}{\textbf{Tumour-free LGG FLAIR slices}} & \textbf{Nick. test} \\',
           r'\cmidrule(lr){3-6}',
           r'\textbf{Model} & \textbf{LGG} & \textbf{Gli.} & \textbf{Men.} & \textbf{Pit.} & \textbf{NoT.} & '
           r'\textbf{Gli. recall} \\'], prob_rows,
          note='LGG: whether LGG crops were used in training. Gli.--NoT.: percentage of the tumour-free LGG FLAIR slices (never used for training) assigned to '
               'each class by the full pipeline. A model that relies on the FLAIR appearance of the glioma-only LGG '
               "source rather than on tumour features would label these slices ``glioma''.")

# seeds (optional) -----------------------------------------------------------------------------------
seed_runs = [r for r in ['main', 'main_seed1', 'main_seed2'] if r in RUNS]
if len(seed_runs) > 1:
    for t in TESTS:
        accs = [100 * (pred_ttaT(r, t)[0].argmax(1) == pred_ttaT(r, t)[1]).mean() for r in seed_runs]
        setm(f'seed_{t}_mean', float(np.mean(accs)), '{:.2f}'); setm(f'seed_{t}_sd', float(np.std(accs, ddof=1)), '{:.2f}')
    setm('n_seeds', len(seed_runs))

In [ ]:
# ── 18. Grad-CAM++ localisation against BRISC expert tumour masks ─────────────────────────────────────
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

cam_model = fresh_model(pretrained=False)
cam_model.load_state_dict(torch.load(DIRS['checkpoints'] / 'main_selected.pt', map_location=DEVICE))
cam_model.eval()
GRID = CFG['img_size'] // 16
def reshape_transform(t, h=GRID, w=GRID):
    return t[:, 1:, :].reshape(t.size(0), h, w, t.size(2)).permute(0, 3, 1, 2)
cam = GradCAMPlusPlus(model=cam_model, target_layers=[cam_model.backbone.blocks[-1].norm1],
                      reshape_transform=reshape_transform)

SZ_ = CFG['img_size']; _yy, _xx = np.mgrid[0:SZ_, 0:SZ_]
CENTRE = np.exp(-((_xx - SZ_ / 2) ** 2 + (_yy - SZ_ / 2) ** 2) / (2 * (0.25 * SZ_) ** 2))
_tol = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * CFG['cam_pg_tolerance_px'] + 1,) * 2)

def cam_metrics(c, mask, brain):
    """c: CAM in [0,1] (384²); mask: tumour mask (bool); brain: foreground mask (bool)."""
    brain = brain | mask            # the tumour lies inside the head even where the Otsu mask missed it (zoomed slices)
    md = cv2.dilate(mask.astype(np.uint8), _tol).astype(bool)
    py, px = np.unravel_index(int(np.argmax(c)), c.shape)
    e_brain = float((c * brain).sum()); area = mask.sum() / max(brain.sum(), 1)
    e_in = float((c * mask).sum()) / e_brain if e_brain > 0 else np.nan
    g_in = float((CENTRE * mask).sum() / (CENTRE * brain).sum())
    b = c >= 0.5
    return dict(pg_hit=bool(md[py, px]), energy_in=e_in, enrich=e_in / area if area > 0 else np.nan,
                iou=float((b & mask).sum() / max((b | mask).sum(), 1)), mass_brain=e_brain / max(float(c.sum()), 1e-8),
                area_frac=float(area), centre_hit=bool(md[SZ_ // 2, SZ_ // 2]), centre_energy_in=g_in,
                centre_enrich=g_in / area if area > 0 else np.nan)

def run_cam(paths, targets):
    out = []
    for s in range(0, len(paths), CFG['cam_batch']):
        x = torch.stack([val_transform(Image.open(p).convert('RGB')) for p in paths[s:s + CFG['cam_batch']]]).to(DEVICE)
        out.append(cam(input_tensor=x, targets=[ClassifierOutputTarget(int(c)) for c in targets[s:s + CFG['cam_batch']]]))
    return np.concatenate(out) if out else np.zeros((0, SZ_, SZ_))

def pred_table(t):
    L, y, u = sel_logits('main', t); p = probs_for(L, 'ttaT', T0)
    return pd.DataFrame(dict(uid=u, y=y, pred=p.argmax(1), conf=p.max(1)))

PB = pred_table('brisc').merge(U[['uid', 'pre_ws', 'mask_path', 'plane']], on='uid')
tum = PB[(PB.y < 3) & PB.mask_path.ne('')].reset_index(drop=True)
maps = run_cam(tum.pre_ws.tolist(), tum.pred.tolist())
cam_rows = []
for i, r in tum.iterrows():
    mask = cv2.resize((np.array(Image.open(r.mask_path).convert('L')) > 0).astype(np.uint8), (SZ_, SZ_),
                      interpolation=cv2.INTER_NEAREST).astype(bool)
    img = np.array(Image.open(r.pre_ws).convert('L').resize((SZ_, SZ_)))
    if mask.sum() == 0: continue
    cam_rows.append(dict(uid=r.uid, y=r.y, pred=r.pred, correct=r.y == r.pred, plane=r.plane,
                         **cam_metrics(maps[i], mask, foreground_mask(img.astype(np.float32)))))
CAM = pd.DataFrame(cam_rows); CAM.to_csv(DIRS['results'] / 'gradcam_localisation.csv', index=False)

def cam_summary(d):
    k, n = int(d.pg_hit.sum()), len(d); lo, hi = wilson(k, n)
    q = lambda c: (d[c].median(), d[c].quantile(.25), d[c].quantile(.75))
    return dict(n=n, pg=k / n if n else np.nan, pg_lo=lo, pg_hi=hi, centre_pg=d.centre_hit.mean() if n else np.nan,
                energy=q('energy_in'), enrich=q('enrich'), iou=q('iou'), mass_brain=q('mass_brain'),
                centre_enrich=d.centre_enrich.median() if n else np.nan)
rows = []; n_cls_rows = 0
groups = [(CLASSES[c], CAM[(CAM.y == c) & CAM.correct]) for c in range(3)] + \
         [('All correct', CAM[CAM.correct]), ('Misclassified', CAM[~CAM.correct]), ('All', CAM)]
for lab, d in groups:
    if not len(d): continue
    s_ = cam_summary(d)
    n_cls_rows += lab in CLASSES
    rows.append(' & '.join([lab, f"{s_['n']:,}", f"{pc(s_['pg'], 1)} {ci(s_['pg_lo'], s_['pg_hi'], 1)}", pc(s_['centre_pg'], 1),
                            f"{s_['energy'][0]:.2f}", f"{s_['enrich'][0]:.1f}", f"{s_['centre_enrich']:.1f}", f"{s_['iou'][0]:.2f}"]))
    key = {'All correct': 'correct', 'Misclassified': 'wrong', 'All': 'all'}.get(lab, lab.lower()[:3])
    setm(f'cam_{key}_n', s_['n'], '{:,}'); setm(f'cam_{key}_pg', 100 * s_['pg'], '{:.1f}')
    setm(f'cam_{key}_pg_lo', 100 * s_['pg_lo'], '{:.1f}'); setm(f'cam_{key}_pg_hi', 100 * s_['pg_hi'], '{:.1f}')
    setm(f'cam_{key}_centre_pg', 100 * s_['centre_pg'], '{:.1f}'); setm(f'cam_{key}_energy', s_['energy'][0], '{:.2f}')
    setm(f'cam_{key}_energy_q1', s_['energy'][1], '{:.2f}'); setm(f'cam_{key}_energy_q3', s_['energy'][2], '{:.2f}')
    setm(f'cam_{key}_enrich', s_['enrich'][0], '{:.1f}'); setm(f'cam_{key}_centre_enrich', s_['centre_enrich'], '{:.1f}')
    setm(f'cam_{key}_iou', s_['iou'][0], '{:.2f}'); setm(f'cam_{key}_mass_brain', s_['mass_brain'][0], '{:.2f}')
rows.insert(n_cls_rows, r'\midrule')
write_tab('tab_gradcam', 'l r c r r r r r',
          [r' & & \multicolumn{2}{c}{\textbf{Pointing game (\%)}} & \textbf{Energy} & \multicolumn{2}{c}{\textbf{Enrichment}} & \\',
           r'\cmidrule(lr){3-4} \cmidrule(lr){6-7}',
           r'\textbf{Group} & $n$ & \textbf{CAM [95\% CI]} & \textbf{Centre} & \textbf{in mask} & \textbf{CAM} & '
           r'\textbf{Centre} & \textbf{IoU} \\'], rows,
          note=f'BRISC test images with expert tumour masks. Grad-CAM++ (last encoder block, predicted class, single identity '
               f"view). Pointing game: CAM maximum within {CFG['cam_pg_tolerance_px']}\\,px of the mask; ``Centre'': the same "
               'metric for a trivial saliency map peaked at the image centre (isotropic Gaussian). Energy in mask: fraction '
               'of the within-head CAM mass inside the tumour mask; enrichment: that fraction divided by the mask\'s share '
               'of the head area (1 = no better than uniform). Energy, enrichment and IoU (CAM $\\ge$ 0.5 vs mask) are '
               'medians over images; per-image values are released with the code.')

In [ ]:
# ── 19. Compute and latency ────────────────────────────────────────────────────────────────────────────
from torch.utils.flop_counter import FlopCounterMode

def count_flops(model, shape):
    x = torch.randn(*shape, device=DEVICE)
    with torch.no_grad(), FlopCounterMode(display=False) as fc: model(x)
    return float(fc.get_total_flops())

def bench(fn, warm=None, iters=None):
    warm, iters = warm or CFG['bench_warmup'], iters or CFG['bench_iters']
    for _ in range(warm): fn()
    torch.cuda.synchronize(); ts = []
    for _ in range(iters):
        t0 = time.perf_counter(); fn(); torch.cuda.synchronize(); ts.append(1000 * (time.perf_counter() - t0))
    return float(np.median(ts)), float(np.percentile(ts, 95))

def peak_mb(fn, weights_mb):
    """Weights + the extra memory allocated while running fn (inputs already resident are not counted)."""
    torch.cuda.synchronize(); torch.cuda.empty_cache(); base = torch.cuda.memory_allocated()
    torch.cuda.reset_peak_memory_stats(); fn(); torch.cuda.synchronize()
    return weights_mb + (torch.cuda.max_memory_allocated() - base) / 2 ** 20

def weights_mb_of(models):
    return sum(p.numel() * p.element_size() for m in models for p in m.parameters()) / 2 ** 20

del cam; gc.collect(); torch.cuda.empty_cache()
bm = cam_model.eval()
n_params = sum(p.numel() for p in bm.parameters())
flop_model = fresh_model(pretrained=False).eval()          # explicit attention so QKᵀ and AV matmuls are counted
for b in flop_model.backbone.blocks:
    if hasattr(b.attn, 'fused_attn'): b.attn.fused_attn = False
fl_pass = count_flops(flop_model, (1, 3, SZ_, SZ_)); del flop_model; gc.collect(); torch.cuda.empty_cache()
Ntok, Dm, Lb = (SZ_ // 16) ** 2 + 1, 768, 12
macs_analytic = Lb * (Ntok * Dm * 3 * Dm + 2 * Ntok * Ntok * Dm + Ntok * Dm * Dm + 2 * Ntok * Dm * 4 * Dm) \
                + (Ntok - 1) * Dm * (16 * 16 * 3)
print(f'FLOPs/pass measured {fl_pass / 1e9:.1f} G | analytic 2×MACs {2 * macs_analytic / 1e9:.1f} G')

x1 = torch.randn(1, 3, SZ_, SZ_, device=DEVICE); x6 = torch.randn(6, 3, SZ_, SZ_, device=DEVICE)
x14 = torch.randn(14, 3, SZ_, SZ_, device=DEVICE); x64 = torch.randn(64, 3, SZ_, SZ_, device=DEVICE)
def fwd(x):
    with torch.no_grad(), torch.autocast('cuda', dtype=AMP_DTYPE): return bm(x).float().softmax(-1).mean(0)
lat1 = bench(lambda: fwd(x1)); lat6 = bench(lambda: fwd(x6)); lat14 = bench(lambda: fwd(x14))
lat14s = bench(lambda: [fwd(x1) for _ in range(14)], iters=max(10, CFG['bench_iters'] // 5))
thr64 = 64 / (bench(lambda: fwd(x64), iters=max(10, CFG['bench_iters'] // 5))[0] / 1000)
w_mb = weights_mb_of([bm])
mem1, mem14 = peak_mb(lambda: fwd(x1), w_mb), peak_mb(lambda: fwd(x14), w_mb)
img0 = Image.open(U[U.role == 'test_nick'].pre_ws.iloc[0]).convert('RGB')
e2e1 = bench(lambda: fwd(TTA_VIEWS[0](img0).unsqueeze(0).to(DEVICE)), iters=max(10, CFG['bench_iters'] // 2))
e2e14 = bench(lambda: fwd(torch.stack([v(img0) for v in TTA_VIEWS]).to(DEVICE)), iters=max(10, CFG['bench_iters'] // 5))
cpu_name = next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), platform.processor())
cpu_m = BrainTumorViT(pretrained=False).eval(); xc = torch.randn(1, 3, SZ_, SZ_)
with torch.no_grad():
    cpu_m(xc); t0 = time.perf_counter()
    for _ in range(CFG['bench_cpu_iters']): cpu_m(xc)
cpu_lat = 1000 * (time.perf_counter() - t0) / CFG['bench_cpu_iters']; del cpu_m

setm('params_total', n_params, '{:,}'); setm('params_m', n_params / 1e6, '{:.1f}')
setm('gflops_pass', fl_pass / 1e9, '{:.1f}'); setm('gmacs_pass', fl_pass / 2e9, '{:.1f}')
setm('gflops_k6', 6 * fl_pass / 1e9, '{:.0f}'); setm('gflops_k14', 14 * fl_pass / 1e9, '{:.0f}')
setm('tflops_k14', 14 * fl_pass / 1e12, '{:.2f}')
setm('lat1_ms', lat1[0], '{:.1f}'); setm('lat6_ms', lat6[0], '{:.1f}'); setm('lat14_ms', lat14[0], '{:.1f}')
setm('lat14s_ms', lat14s[0], '{:.1f}'); setm('lat_e2e1_ms', e2e1[0], '{:.1f}'); setm('lat_e2e14_ms', e2e14[0], '{:.1f}')
setm('thr64', thr64, '{:.0f}'); setm('mem1_mb', mem1, '{:.0f}'); setm('mem14_mb', mem14, '{:.0f}')
setm('weights_mb', w_mb, '{:.0f}'); setm('cpu_lat1_ms', cpu_lat, '{:.0f}'); setm('cpu_name', cpu_name)
setm('comp_size', CFG['comparator_size'])

# comparator backbones (their own papers report no FLOPs/latency; measured here with the same harness)
COMP = {'BRAIN-META': ['densenet121', 'densenet169', 'densenet201', 'mobilenetv1_100', 'mobilenetv2_100',
                       'efficientnet_b0', 'efficientnet_b1', 'efficientnet_b4', 'inception_v3', 'legacy_xception'],
        'BTdiagAI': ['inception_v3', 'densenet121', 'legacy_xception']}
ALT = {'mobilenetv1_100': ['mobilenetv1_100', 'mobilenetv1_100h'], 'legacy_xception': ['legacy_xception', 'xception']}
def make_timm(name):
    for c_ in ALT.get(name, [name]):
        try: return timm.create_model(c_, pretrained=False, num_classes=4).to(DEVICE).eval(), c_
        except Exception: continue
    return None, None
CS = CFG['comparator_size']; comp_res = {}
for grp, names in COMP.items():
    ms, used, P_, Fl_, Lt_ = [], [], 0, 0.0, 0.0
    for nm in names:
        m_, u_ = make_timm(nm)
        if m_ is None: print('⚠ comparator not available in timm:', nm); continue
        ms.append(m_); used.append(u_); P_ += sum(p.numel() for p in m_.parameters())
        Fl_ += count_flops(m_, (1, 3, CS, CS))
        xs_ = torch.randn(1, 3, CS, CS, device=DEVICE)
        def f_(m=m_, x=xs_):
            with torch.no_grad(), torch.autocast('cuda', dtype=AMP_DTYPE): return m(x)
        Lt_ += bench(f_, iters=max(10, CFG['bench_iters'] // 2))[0]
    xs_ = torch.randn(1, 3, CS, CS, device=DEVICE)
    def all_(ms=ms, x=xs_):
        with torch.no_grad(), torch.autocast('cuda', dtype=AMP_DTYPE):
            for m in ms: m(x)
    comp_res[grp] = dict(models=used, params=P_, flops=Fl_, lat=Lt_, mem=peak_mb(all_, weights_mb_of(ms)))
    key = {'BRAIN-META': 'bm', 'BTdiagAI': 'bt'}[grp]
    setm(f'{key}_params_m', P_ / 1e6, '{:.1f}'); setm(f'{key}_gflops', Fl_ / 1e9, '{:.1f}')
    setm(f'{key}_lat_ms', Lt_, '{:.1f}'); setm(f'{key}_mem_mb', comp_res[grp]['mem'], '{:.0f}')
    setm(f'{key}_n_models', len(used))
    del ms; gc.collect(); torch.cuda.empty_cache()

acc_n = RES[('nick', 'ttaT')]['acc']; acc_n1 = RES[('nick', 'singleT')]['acc']; acc_n6 = RES[('nick', 'k6T')]['acc']
crow = [' & '.join(['BTFuseViT, single view ($K{=}1$)', f'{n_params / 1e6:.1f}', f'{fl_pass / 1e9:.1f}', f'{lat1[0]:.1f}',
                    f'{mem1:.0f}', pc(acc_n1)]),
        ' & '.join(['BTFuseViT, TTA $K{=}6$', f'{n_params / 1e6:.1f}', f'{6 * fl_pass / 1e9:.0f}', f'{lat6[0]:.1f}',
                    '--', pc(acc_n6)]),
        ' & '.join(['BTFuseViT, TTA $K{=}14$ (full)', f'{n_params / 1e6:.1f}', f'{14 * fl_pass / 1e9:.0f}',
                    f'{lat14[0]:.1f}', f'{mem14:.0f}', pc(acc_n)])]
for grp, rep in [('BRAIN-META', '97.10$^{a}$'), ('BTdiagAI', '99.80$^{b}$')]:
    r_ = comp_res.get(grp)
    if r_: crow.append(' & '.join([f'{grp} backbones only ($\\times${len(r_["models"])})', f'{r_["params"] / 1e6:.1f}',
                                   f'{r_["flops"] / 1e9:.1f}', f'{r_["lat"]:.1f}', f'{r_["mem"]:.0f}', rep]))
write_tab('tab_compute', 'p{3.9cm} r r r r r',
          [r'\textbf{Model / inference} & \textbf{Params} & \textbf{GFLOPs} & \textbf{Latency} & \textbf{Peak mem.} & \textbf{Acc.} \\',
           r' & (M) & per image & (ms) & (MB) & (\%) \\'], crow,
          note=f'Measured on one {tex_escape(GPU_NAME)}, batch = 1 image (TTA views batched), bf16, model only; '
               f'GFLOPs = 2 $\\times$ multiply--accumulates, attention included. Peak memory = fp32 weights + activations. '
               f'Comparator rows are our measurements of the published CNN backbones at {CS}$\\times${CS} with standard timm '
               'implementations and exclude their ViT heads, meta-learners and feature-selection stages, so they are '
               'lower bounds. Accuracy: ours on the Nickparvar test set; $^{a}$BRAIN-META as published on Nickparvar v1 '
               '(1,311 test images); $^{b}$BTdiagAI as published on 1,000 images of an augmentation-balanced Nickparvar v1 '
               'split. Accuracies are not comparable across rows with different test data.')
save_json(dict(params=n_params, flops_pass=fl_pass, macs_analytic=macs_analytic, lat1=lat1, lat6=lat6, lat14=lat14,
               lat14_sequential=lat14s, e2e1=e2e1, e2e14=e2e14, thr64=thr64, mem1=mem1, mem14=mem14, weights_mb=w_mb,
               cpu_name=cpu_name, cpu_lat_ms=cpu_lat, gpu=GPU_NAME, comparators=comp_res), DIRS['results'] / 'compute.json')
print(f'params {n_params:,} | {fl_pass / 1e9:.1f} GFLOPs/pass | K=14 {14 * fl_pass / 1e9:.0f} GFLOPs | '
      f'latency 1 view {lat1[0]:.1f} ms, 14 views {lat14[0]:.1f} ms | peak {mem1:.0f}/{mem14:.0f} MB | CPU {cpu_lat:.0f} ms')

FLOPs/pass measured 111.0 G | analytic 2×MACs 111.0 G
params 86,619,652 | 111.0 GFLOPs/pass | K=14 1554 GFLOPs | latency 1 view 2.5 ms, 14 views 9.1 ms | peak 521/696 MB | CPU 75 ms


In [ ]:
# ── 20. Figures (sized for the 13.1 cm text width of the Springer Nature template; fonts 7–8 pt) ──────
plt.rcParams.update({'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 7,
                     'ytick.labelsize': 7, 'legend.fontsize': 7, 'savefig.dpi': 300, 'pdf.fonttype': 42,
                     'axes.spines.top': False, 'axes.spines.right': False})
FIG_W = 5.1
COLORS = ['#c0392b', '#2471a3', '#239b56', '#b9770e']
def savefig(fig, name):
    fig.savefig(DIRS['figures'] / f'{name}.pdf', bbox_inches='tight'); fig.savefig(DIRS['figures'] / f'{name}.png', bbox_inches='tight')
    plt.close(fig)

# confusion matrices
fig, axes = plt.subplots(1, len(TESTS), figsize=(FIG_W, 1.95), constrained_layout=True)
axes = np.atleast_1d(axes); short = ['Gli', 'Men', 'Pit', 'NoT']
for ax, t in zip(axes, TESTS):
    cm = RES[(t, 'ttaT')]['cm']; cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
    ax.imshow(cmn, cmap='Blues', vmin=0, vmax=1)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{cm[i, j]}', ha='center', va='center', fontsize=7, color='white' if cmn[i, j] > 0.6 else 'black')
    ax.set_xticks(range(4)); ax.set_yticks(range(4)); ax.set_xticklabels(short); ax.set_yticklabels(short)
    ax.set_xlabel('Predicted'); ax.set_title(f"{TEST_NAMES[t]} (n={cm.sum():,})")
axes[0].set_ylabel('True')
savefig(fig, 'fig_confusion')

# ROC curves
roc_t = [t for t in TESTS if t in ('nick', 'brisc')]
fig, axes = plt.subplots(1, len(roc_t), figsize=(FIG_W, 2.4), constrained_layout=True); axes = np.atleast_1d(axes)
for ax, t in zip(axes, roc_t):
    L, y, _ = sel_logits('main', t); p = probs_for(L, 'ttaT', T0)
    for c in range(4):
        if (y == c).sum() == 0: continue
        fpr, tpr, _ = roc_curve(y == c, p[:, c])
        ax.plot(fpr, tpr, color=COLORS[c], lw=1.2, label=f"{CLASSES[c]} (AUC {RES[(t, 'ttaT')]['AUC'][c]:.4f})")
    ax.plot([0, 1], [0, 1], ls='--', lw=0.7, color='grey')
    ax.set_xlabel('False-positive rate'); ax.set_ylabel('True-positive rate'); ax.set_title(TEST_NAMES[t])
    ax.legend(loc='lower right', frameon=False)
savefig(fig, 'fig_roc')

# reliability diagrams
def reliability(ax, p, y, label, color):
    conf, acc = p.max(1), (p.argmax(1) == y)
    edges = np.linspace(0, 1, CFG['ece_bins'] + 1); xs, ys, ns = [], [], []
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum(): xs.append(conf[m].mean()); ys.append(acc[m].mean()); ns.append(m.sum())
    ax.plot(xs, ys, marker='o', ms=3, lw=1, color=color, label=f'{label} (ECE {100 * ece_score(p, y):.2f}%)')
fig, axes = plt.subplots(1, len(roc_t), figsize=(FIG_W, 2.95), constrained_layout=True); axes = np.atleast_1d(axes)
for ax, t in zip(axes, roc_t):
    L, y, _ = sel_logits('main', t)
    for mode, col, lab in [('single', COLORS[1], 'single view, T=1'), ('tta', COLORS[3], 'TTA, T=1'),
                           ('ttaT', COLORS[0], f'TTA, T*={T0:.2f}')]:
        reliability(ax, probs_for(L, mode, T0), y, lab, col)
    ax.plot([0, 1], [0, 1], ls='--', lw=0.7, color='grey'); ax.set_xlim(0.2, 1.01); ax.set_ylim(0.2, 1.01)
    ax.set_xlabel('Confidence'); ax.set_ylabel('Accuracy'); ax.set_title(TEST_NAMES[t])
    ax.legend(frameon=False, loc='upper center', bbox_to_anchor=(0.5, -0.24), fontsize=6.5)   # below the axes: never over the curves
savefig(fig, 'fig_reliability')

# training curves (main run; the validation split, not a test set)
Hh = pd.DataFrame(RUNS['main']['history'])
fig, axes = plt.subplots(1, 2, figsize=(FIG_W, 2.1), constrained_layout=True)
we, sw = CFG['warmup_epochs'], CFG['warmup_epochs'] + CFG['swa_start']
for ax, (a, b, ttl) in zip(axes, [('tr_loss', 'val_loss', 'ASL loss'), ('tr_acc', 'val_acc', 'Accuracy (%)')]):
    ax.axvspan(0.5, we + 0.5, color='#eeeeee'); ax.axvspan(sw + 0.5, len(Hh) + 0.5, color='#e8f0fb')
    ax.plot(Hh.epoch, Hh[a], color=COLORS[1], lw=1.2, label='training'); ax.plot(Hh.epoch, Hh[b], color=COLORS[0], lw=1.2, label='validation')
    ax.set_xlabel('Epoch'); ax.set_title(ttl); ax.set_xlim(0.5, len(Hh) + 0.5)
axes[1].axvline(S0['best_epoch'], color='grey', lw=0.6, ls=':')
axes[0].legend(frameon=False); axes[1].text(we / 2 + 0.5, axes[1].get_ylim()[0] + 2, 'P1', ha='center', fontsize=7)
axes[1].text(len(Hh) - 2, axes[1].get_ylim()[0] + 2, 'SWA', ha='center', fontsize=7)
savefig(fig, 'fig_training')

# per-view accuracy (supplementary)
fig, ax = plt.subplots(figsize=(FIG_W, 2.2), constrained_layout=True); w = 0.38
for i, t in enumerate(roc_t):
    L, y, _ = sel_logits('main', t)
    accs = [100 * (probs_for(L, f'view{k}', T0).argmax(1) == y).mean() for k in range(CFG['tta_k'])]
    ax.bar(np.arange(CFG['tta_k']) + (i - 0.5) * w, accs, w, color=COLORS[i + 1], label=TEST_NAMES[t])
    ax.axhline(100 * RES[(t, 'ttaT')]['acc'], color=COLORS[i + 1], ls='--', lw=0.8)
ax.set_xticks(range(CFG['tta_k'])); ax.set_xticklabels(range(1, CFG['tta_k'] + 1)); ax.set_xlabel('TTA view (Table S2)')
ax.set_ylabel('Accuracy (%)'); lo_ = min(ax.get_ylim()[1] - 30, 70); ax.set_ylim(lo_, 100.5); ax.legend(frameon=False, ncol=2)
savefig(fig, 'fig_perview')

# duplicate examples (supplementary)
cross = CONF[(CONF.src_i != CONF.src_j) & (CONF.src_i.isin(EVAL_SOURCES) | CONF.src_j.isin(EVAL_SOURCES))
             & ~CONF.src_i.eq('lgg') & ~CONF.src_j.eq('lgg')]
near = PAIRS[~PAIRS.dup & (PAIRS.ncc >= CFG['dup_ncc']) & (PAIRS.bp >= CFG['dup_bp'] - 0.1) & (PAIRS.src_i != PAIRS.src_j)]
ex = pd.concat([cross.sample(min(6, len(cross)), random_state=0).assign(kind='confirmed'),
                cross.nsmallest(min(3, len(cross)), 'bp').assign(kind='confirmed (borderline)'),
                near.nlargest(min(3, len(near)), 'bp').assign(kind='rejected (below threshold)')])
if len(ex):
    npair_cols = 3; nrows_ = int(math.ceil(len(ex) / npair_cols))
    fig, axes = plt.subplots(nrows_, 2 * npair_cols, figsize=(FIG_W, 1.05 * nrows_ + 0.1), constrained_layout=True)
    axes = np.atleast_2d(axes)
    for ax in axes.ravel(): ax.axis('off')
    for k, row in enumerate(ex.itertuples()):
        r_, c0 = divmod(k, npair_cols); c0 *= 2
        tag = {'confirmed': 'linked', 'confirmed (borderline)': 'linked (borderline)',
               'rejected (below threshold)': 'not linked'}[row.kind]
        for c, (i, s_, l) in enumerate([(row.i, row.src_i, row.lab_i), (row.j, row.src_j, row.lab_j)]):
            ax = axes[r_, c0 + c]; ax.imshow(read_gray(SCR.path[i], s_ == 'lgg'), cmap='gray')
            extra = tag if c == 0 else f'NCC {row.ncc:.3f} BP {row.bp:.3f}'
            ax.set_title(f'{SRC_LABEL[s_]} · {["Gli.", "Men.", "Pit.", "No tum."][l]}\n{extra}', fontsize=5)
    savefig(fig, 'fig_overlap_examples')

# Grad-CAM++ figures: correct cases (with expert masks) and ALL kinds of errors
cam = GradCAMPlusPlus(model=cam_model, target_layers=[cam_model.backbone.blocks[-1].norm1],
                      reshape_transform=reshape_transform)       # run_cam() uses this object
def overlay(ax, path, cmap_, mask=None, title=''):
    g = np.array(Image.open(path).convert('L').resize((SZ_, SZ_))) / 255.0
    ax.imshow(g, cmap='gray'); ax.imshow(cmap_, cmap='jet', alpha=0.40, vmin=0, vmax=1)
    if mask is not None and mask.any(): ax.contour(mask, levels=[0.5], colors='white', linewidths=0.8)
    py, px = np.unravel_index(int(np.argmax(cmap_)), cmap_.shape); ax.plot(px, py, marker='+', color='white', ms=6, mew=1.2)
    ax.set_title(title, fontsize=6.5); ax.axis('off')
def mask384(p): return cv2.resize((np.array(Image.open(p).convert('L')) > 0).astype(np.uint8), (SZ_, SZ_), interpolation=cv2.INTER_NEAREST).astype(bool)
rng_ = np.random.default_rng(0); pick = []
for c in range(3):
    d = CAM[(CAM.y == c) & CAM.correct]
    pick += list(d.uid.values[rng_.choice(len(d), min(4, len(d)), replace=False)]) if len(d) else []
if pick:
    sub = tum.set_index('uid').loc[pick]
    mp_ = run_cam(sub.pre_ws.tolist(), sub.pred.tolist())
    fig, axes = plt.subplots(3, 4, figsize=(FIG_W, 4.0), constrained_layout=True)
    for ax in axes.ravel(): ax.axis('off')
    for k, (uid, r) in enumerate(sub.iterrows()):
        row_ = CAM.set_index('uid').loc[uid]
        overlay(axes.ravel()[k], r.pre_ws, mp_[k], mask384(r.mask_path),
                f"{CLASSES[r.y][:4]}. {'hit' if row_.pg_hit else 'miss'}, E {row_.energy_in:.2f}")
    savefig(fig, 'fig_gradcam')
errs = []                                   # up to 4 most confident errors per test set (failure cases, R2.8)
for t in TESTS:
    d = pred_table(t).merge(U[['uid', 'pre_ws', 'mask_path']], on='uid'); d = d[d.y != d.pred]
    errs.append(d.sort_values('conf', ascending=False).head(4).assign(t=t))
errs = pd.concat(errs) if errs else pd.DataFrame()
if len(errs):
    errs = errs.reset_index(drop=True)
    mp_ = run_cam(errs.pre_ws.tolist(), errs.pred.tolist())
    nr = int(math.ceil(len(errs) / 4))
    fig, axes = plt.subplots(nr, 4, figsize=(FIG_W, 1.35 * nr + 0.2), constrained_layout=True)
    axes = np.atleast_2d(axes)
    for ax in axes.ravel(): ax.axis('off')
    for k, r in enumerate(errs.itertuples()):
        overlay(axes.ravel()[k], r.pre_ws, mp_[k], mask384(r.mask_path) if r.mask_path else None,
                f"{ {'nick': 'Nick.', 'brisc': 'BRISC', 'pmram': 'PMRAM'}[r.t]}: {CLASSES[r.y][:4]}→{CLASSES[r.pred][:4]} ({r.conf:.2f})")
    savefig(fig, 'fig_gradcam_errors')
    setm('n_error_panels', len(errs))
del cam; gc.collect(); torch.cuda.empty_cache()

# training-history table (supplementary)
hrows = []
for h in RUNS['main']['history']:
    star = r'$^{\ast}$' if h['epoch'] == S0['best_epoch'] else ''
    hrows.append(' & '.join([f"{h['epoch']}{star}", h['phase'], f"{h['tr_loss']:.4f}", f"{h['tr_acc']:.2f}",
                             f"{h['val_loss']:.4f}", f"{h['val_acc']:.2f}"]))
write_tab('tab_history_supp', 'r l r r r r',
          [r'\textbf{Epoch} & \textbf{Phase} & \textbf{Train loss} & \textbf{Train acc. (\%)} & '
           r'\textbf{Val. loss} & \textbf{Val. acc. (\%)} \\'], hrows,
          note=f"Validation = the grouped validation split (n = {S0['n_val']:,}); no test image is used during training. "
               f"$^{{\\ast}}$Best single checkpoint (validation accuracy {S0['val_acc_best']:.2f}\\%); SWA model: "
               f"{S0['val_acc_swa']:.2f}\\%; selected: {'SWA' if S0['selected'] == 'swa' else 'best checkpoint'}; "
               f"$T^{{*}}={T0:.4f}$.")

In [ ]:
# ── 21. Write macros + results and package everything for the manuscript ───────────────────────────────
MAC['smoke_test'] = 'yes' if SMOKE_TEST else 'no'
with open(DIRS['generated'] / 'results_macros.tex', 'w') as f:
    f.write('% Auto-generated by BTFuseViT_v6.ipynb -- do not edit by hand.\n')
    f.write(f'% generated {time.strftime("%Y-%m-%d %H:%M")} | SMOKE_TEST={SMOKE_TEST}\n')
    for k in sorted(MAC): f.write(f'\\setres{{{k}}}{{{MAC[k]}}}\n')
save_json(dict(macros=MAC, cfg=CFG, run_plan=RUN_PLAN, smoke_test=SMOKE_TEST,
               runs={k: v['summary'] for k, v in RUNS.items()}), DIRS['results'] / 'results.json')
zp = PERSIST / 'BTFuseViT_v6_outputs.zip'
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as z:
    for sub in ['generated', 'figures', 'results']:
        for f in (PERSIST / sub).rglob('*'):
            if f.is_file(): z.write(f, f.relative_to(PERSIST))
    z.write(DIRS['manifests'] / 'partition_manifest.csv', 'manifests/partition_manifest.csv')
    for f in DIRS['runs'].rglob('*'):
        if f.is_file(): z.write(f, f.relative_to(PERSIST))
print(f'✓ {len(MAC)} macros | outputs → {zp} ({zp.stat().st_size / 1e6:.1f} MB)')
print('All tables, figures and numbers of the paper are in BTFuseViT_v6_outputs.zip.' if not SMOKE_TEST else
      'SMOKE TEST PASSED — set SMOKE_TEST = False in the config cell and Run all for the real experiments.')

✓ 773 macros | outputs → /content/drive/MyDrive/BTFuseViT_v6/BTFuseViT_v6_outputs.zip (13.7 MB)
All tables, figures and numbers of the paper are in BTFuseViT_v6_outputs.zip.
